> **Public snapshot note**
> 이 노트북은 원래 연구 실행 흐름을 보존한 archival artifact입니다.
> 공개 스냅샷에서는 raw PDF, processed corpus, `kg_gen/graphrag_workspace/input`, `kg_gen/graphrag_workspace/output`,
> `kg_gen/snu_kg_output/graphrag_workspace/output*` 등 bulk/source-derived 자산을 의도적으로 제외했습니다.
> 따라서 아래의 일부 경로 설명은 **원래 실험 환경 기준**이며, 현재 public snapshot과 1:1로 대응하지 않을 수 있습니다.


# GraphRAG QA Dataset Generation

이 노트북은 GraphRAG 출력을 기반으로 5가지 유형의 QA 데이터셋을 생성합니다:
1. Multi-hop reasoning (다중 홉 추론)
2. Community synthesis (커뮤니티 기반 합성)
3. Cross-context integration (교차 컨텍스트 통합)
4. Implicit relationships (암시적 관계)
5. Causal chains (인과 관계 체인)

## Cell 1: 라이브러리 임포트

In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
from pathlib import Path
import json
from collections import defaultdict
import pickle
from typing import Dict, List, Tuple, Set
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 로드 완료")

## Cell 2: 경로 설정

In [ ]:
# 기본 경로 설정
base_path = Path("<PROJECT_ROOT>")
kg_path = base_path / "kg_gen/snu_kg_output/graphrag_workspace/output_after_kggen"
output_path = base_path / "rag_dataset/kg_qa_datasets"

# 경로 확인
print(f"Base path: {base_path}")
print(f"KG path exists: {kg_path.exists()}")
print(f"Output path exists: {output_path.exists()}")

# output_path가 없으면 생성
if not output_path.exists():
    output_path.mkdir(parents=True, exist_ok=True)
    print(f"Created output path: {output_path}")

# QA 유형별 디렉토리 확인 및 생성
qa_types = ['multi_hop', 'community_synthesis', 'cross_context', 
            'implicit_relationships', 'causal_chains']
for qa_type in qa_types:
    qa_dir = output_path / qa_type
    if not qa_dir.exists():
        qa_dir.mkdir(parents=True, exist_ok=True)
        print(f"Created {qa_type} directory")
    else:
        print(f"{qa_type} directory exists: True")

## Cell 3: Parquet 파일 로드 및 검증

In [ ]:
# 모든 parquet 파일 로드
data_files = {
    'entities': 'entities.parquet',
    'relationships': 'relationships.parquet',
    'communities': 'communities.parquet',
    'community_reports': 'community_reports.parquet',
    'text_units': 'text_units.parquet',
    'documents': 'documents.parquet',
    'covariates': 'covariates.parquet'
}

data = {}
print("=== Parquet 파일 로드 ===")
for name, filename in data_files.items():
    file_path = kg_path / filename
    if file_path.exists():
        data[name] = pd.read_parquet(file_path)
        print(f"✓ {name}: {data[name].shape} - {filename}")
    else:
        print(f"✗ {name}: 파일을 찾을 수 없음 - {filename}")

# 기본 컬럼 확인
print("\n=== 주요 컬럼 확인 ===")
print(f"entities columns: {list(data['entities'].columns)}")
print(f"relationships columns: {list(data['relationships'].columns)}")

## Cell 4: 데이터 무결성 검증

In [ ]:
print("=== 데이터 무결성 검증 ===")

# 1. relationships의 source/target이 모두 entities.title에 존재하는지
entity_titles = set(data['entities']['title'])
rel_sources = set(data['relationships']['source'])
rel_targets = set(data['relationships']['target'])

source_match = len(rel_sources & entity_titles)
target_match = len(rel_targets & entity_titles)

print(f"\n1. Relationship → Entity 매칭:")
print(f"   - Source 매칭률: {source_match}/{len(rel_sources)} ({source_match/len(rel_sources)*100:.1f}%)")
print(f"   - Target 매칭률: {target_match}/{len(rel_targets)} ({target_match/len(rel_targets)*100:.1f}%)")

# 2. communities의 entity_ids가 모두 entities.id에 존재하는지
entity_ids = set(data['entities']['id'])
all_community_entity_ids = set()
for ids in data['communities']['entity_ids']:
    if ids is not None and len(ids) > 0:
        all_community_entity_ids.update(ids)

community_match = len(all_community_entity_ids & entity_ids)
print(f"\n2. Community → Entity 매칭:")
print(f"   - Entity ID 매칭률: {community_match}/{len(all_community_entity_ids)} ({community_match/len(all_community_entity_ids)*100:.1f}%)")

# 3. text_unit_ids 일관성 검증
text_unit_ids = set(data['text_units']['id'])
entity_text_units = set()
for ids in data['entities']['text_unit_ids']:
    if isinstance(ids, np.ndarray):
        entity_text_units.update(ids)

text_match = len(entity_text_units & text_unit_ids)
print(f"\n3. Entity → Text Unit 매칭:")
print(f"   - Text Unit ID 매칭률: {text_match}/{len(entity_text_units)} ({text_match/len(entity_text_units)*100:.1f}%)")

# 4. 커뮤니티 레벨 분포
print(f"\n4. 커뮤니티 레벨 분포:")
for level in sorted(data['communities']['level'].unique()):
    count = len(data['communities'][data['communities']['level'] == level])
    print(f"   - Level {level}: {count} communities")

print("\n✓ 데이터 무결성 검증 완료")

## Cell 5: 참조 인덱스 구축

In [ ]:
print("=== 빠른 조회를 위한 인덱스 생성 ===")

# 1. Entity title ↔ id 매핑
entity_title_to_id = dict(zip(data['entities']['title'], data['entities']['id']))
entity_id_to_title = {v: k for k, v in entity_title_to_id.items()}
print(f"\n1. Entity 매핑 생성: {len(entity_title_to_id)} 엔티티")

# 2. Text unit 조회 딕셔너리
text_unit_lookup = dict(zip(data['text_units']['id'], data['text_units']['text']))
print(f"2. Text unit 조회 딕셔너리: {len(text_unit_lookup)} 텍스트 유닛")

# 3. Entity → Communities 매핑
entity_to_communities = defaultdict(list)
for _, comm in data['communities'].iterrows():
    for entity_id in comm['entity_ids']:
        entity_to_communities[entity_id].append({
            'community': comm['community'],
            'level': comm['level'],
            'title': comm['title']
        })

print(f"3. Entity → Communities 매핑: {len(entity_to_communities)} 엔티티가 커뮤니티에 속함")

# 4. Community → Report 매핑
community_reports_lookup = {}
for _, report in data['community_reports'].iterrows():
    key = (report['community'], report['level'])
    community_reports_lookup[key] = {
        'title': report['title'],
        'summary': report['summary'],
        'findings': report['findings'] if 'findings' in report else []
    }

print(f"4. Community reports 매핑: {len(community_reports_lookup)} 리포트")

# 5. Document 정보 매핑
document_lookup = {}
text_unit_to_doc = {}
for _, doc in data['documents'].iterrows():
    document_lookup[doc['id']] = {
        'title': doc['title'],
        'metadata': doc['metadata'] if 'metadata' in doc else {}
    }
    for tu_id in doc['text_unit_ids']:
        text_unit_to_doc[tu_id] = doc['id']

print(f"5. Document 매핑: {len(document_lookup)} 문서")

# 6. Entity 빠른 조회를 위한 딕셔너리
entities_by_id = dict(zip(data['entities']['id'], data['entities'].to_dict('records')))
entities_by_title = dict(zip(data['entities']['title'], data['entities'].to_dict('records')))

print("\n✓ 모든 인덱스 구축 완료")

## Cell 6: 기본 통계 출력

In [ ]:
print("=== GraphRAG 데이터 통계 ===")
print(f"\n1. 기본 통계:")
print(f"   - 총 엔티티: {len(data['entities']):,}")
print(f"   - 총 관계: {len(data['relationships']):,}")
print(f"   - 총 문서: {len(data['documents']):,}")
print(f"   - 텍스트 유닛: {len(data['text_units']):,}")
print(f"   - 커뮤니티 레벨: {sorted(data['communities']['level'].unique())}")
print(f"   - Covariates: {len(data['covariates']):,}")

# 가장 중요한 엔티티 (degree 기준)
print("\n2. Top 10 엔티티 (연결도 기준):")
top_entities = data['entities'].nlargest(10, 'degree')[['title', 'type', 'degree', 'frequency']]
for idx, row in top_entities.iterrows():
    print(f"   {row['title']:30} | type: {row['type']:15} | degree: {row['degree']:3} | freq: {row['frequency']:3}")

# 가장 강한 관계들
print("\n3. Top 10 관계 (가중치 기준):")
top_relations = data['relationships'].nlargest(10, 'weight')[['source', 'target', 'weight']]
for idx, row in top_relations.iterrows():
    print(f"   {row['source']:20} → {row['target']:20} | weight: {row['weight']:.0f}")

# 엔티티 타입 분포
print("\n4. 엔티티 타입 분포:")
type_counts = data['entities']['type'].value_counts().head(10)
for etype, count in type_counts.items():
    print(f"   - {etype}: {count} ({count/len(data['entities'])*100:.1f}%)")

# 커뮤니티 크기 분석
print("\n5. 커뮤니티 크기 분석:")
community_sizes = data['communities'].groupby('level')['entity_ids'].apply(lambda x: [len(ids) for ids in x])
for level, sizes in community_sizes.items():
    print(f"   - Level {level}: 평균 {np.mean(sizes):.1f} 엔티티 (최소: {min(sizes)}, 최대: {max(sizes)})")

## Cell 7: 처리된 데이터 저장

In [ ]:
# Phase 2에서 빠른 로드를 위한 데이터 저장
processed_data = {
    'data': data,
    'entity_title_to_id': entity_title_to_id,
    'entity_id_to_title': entity_id_to_title,
    'text_unit_lookup': text_unit_lookup,
    'entity_to_communities': dict(entity_to_communities),
    'community_reports_lookup': community_reports_lookup,
    'document_lookup': document_lookup,
    'text_unit_to_doc': text_unit_to_doc,
    'entities_by_id': entities_by_id,
    'entities_by_title': entities_by_title
}

# pickle로 저장
output_file = output_path / 'processed_graphrag_data.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(processed_data, f)
    
print(f"✓ 처리된 데이터 저장 완료: {output_file}")
print(f"  파일 크기: {output_file.stat().st_size / 1024 / 1024:.2f} MB")

# 저장된 데이터 요약
print("\n저장된 데이터 구조:")
for key in processed_data.keys():
    if key == 'data':
        print(f"  - {key}: {list(processed_data[key].keys())}")
    elif isinstance(processed_data[key], dict):
        print(f"  - {key}: {len(processed_data[key])} items")
    else:
        print(f"  - {key}: {type(processed_data[key])}")

print("\n✓ Phase 1 완료! Phase 2에서 NetworkX 그래프 구축을 진행할 수 있습니다.")

# Phase 2: NetworkX 그래프 구축

이 섹션에서는 GraphRAG 데이터를 NetworkX 그래프로 변환하고 QA 생성을 위한 구조를 준비합니다.

## Cell 8: Phase 1 데이터 로드 및 그래프 초기화

In [ ]:
# Phase 1에서 저장한 데이터 로드
print("=== Phase 1 데이터 로드 중... ===")
with open(output_path / 'processed_graphrag_data.pkl', 'rb') as f:
    processed_data = pickle.load(f)

# 데이터 언패킹
data = processed_data['data']
entity_title_to_id = processed_data['entity_title_to_id']
entity_id_to_title = processed_data['entity_id_to_title']
text_unit_lookup = processed_data['text_unit_lookup']
entity_to_communities = processed_data['entity_to_communities']
community_reports_lookup = processed_data['community_reports_lookup']
document_lookup = processed_data['document_lookup']
text_unit_to_doc = processed_data['text_unit_to_doc']
entities_by_id = processed_data['entities_by_id']
entities_by_title = processed_data['entities_by_title']

print("✓ 데이터 로드 완료")

# NetworkX 그래프 초기화
G = nx.MultiDiGraph()  # 방향성 있고, 중복 엣지 허용
G.graph['name'] = 'Korean Agriculture Knowledge Graph'
G.graph['created_from'] = 'GraphRAG output_after_kggen'
G.graph['node_count'] = len(data['entities'])
G.graph['edge_count'] = len(data['relationships'])

print(f"\n✓ NetworkX MultiDiGraph 초기화 완료")
print(f"  예상 노드 수: {len(data['entities'])}")
print(f"  예상 엣지 수: {len(data['relationships'])}")

## Cell 9: 모든 엔티티를 노드로 추가

In [ ]:
print("=== 노드 추가 중... ===")

# 엔티티를 노드로 추가
for _, entity in data['entities'].iterrows():
    G.add_node(
        entity['title'],  # 노드 ID로 title 사용
        uuid=entity['id'],  # 시스템 ID 보존
        type=entity['type'],
        description=entity['description'],
        frequency=entity['frequency'],
        degree=entity['degree'],
        position=(entity['x'], entity['y']),  # 시각화용 좌표
        text_unit_ids=list(entity['text_unit_ids']) if isinstance(entity['text_unit_ids'], np.ndarray) else [],
        communities={},  # 다음 단계에서 채움
        original_texts=[]  # 다음 단계에서 채움
    )

print(f"✓ {G.number_of_nodes()} 노드 추가 완료")

# 노드 타입별 통계
node_types = defaultdict(int)
for node, attrs in G.nodes(data=True):
    node_types[attrs['type']] += 1

print("\n노드 타입 분포:")
for ntype, count in sorted(node_types.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  - {ntype}: {count}")

## Cell 10: 각 노드에 커뮤니티 정보 추가

In [ ]:
print("=== 커뮤니티 정보 통합 중... ===")

# 각 엔티티가 속한 모든 레벨의 커뮤니티 정보 추가
nodes_with_communities = 0
for entity_id, communities in entity_to_communities.items():
    if entity_id in entity_id_to_title:
        entity_title = entity_id_to_title[entity_id]
        if entity_title in G:
            for comm in communities:
                level = comm['level']
                comm_key = (comm['community'], level)
                
                # 커뮤니티 리포트 정보 가져오기
                report = community_reports_lookup.get(comm_key, {})
                
                G.nodes[entity_title]['communities'][level] = {
                    'id': comm['community'],
                    'title': comm['title'],
                    'report': report
                }
            nodes_with_communities += 1

print(f"✓ {nodes_with_communities} 노드에 커뮤니티 정보 추가 완료")

# 커뮤니티 레벨별 노드 분포 확인
level_distribution = defaultdict(int)
for node, attrs in G.nodes(data=True):
    for level in attrs['communities'].keys():
        level_distribution[level] += 1

print("\n커뮤니티 레벨별 노드 분포:")
for level in sorted(level_distribution.keys()):
    print(f"  - Level {level}: {level_distribution[level]} 노드")

## Cell 11: 모든 관계를 엣지로 추가

In [ ]:
print("=== 엣지 추가 중... ===")

# 관계를 엣지로 추가
edges_added = 0
edges_skipped = 0

for _, rel in data['relationships'].iterrows():
    # source와 target이 모두 그래프에 있는지 확인
    if rel['source'] in G and rel['target'] in G:
        G.add_edge(
            rel['source'],
            rel['target'],
            description=rel['description'],
            weight=rel['weight'],
            text_unit_ids=list(rel['text_unit_ids']) if isinstance(rel['text_unit_ids'], np.ndarray) else [],
            original_texts=[]  # 다음 단계에서 채움
        )
        edges_added += 1
    else:
        edges_skipped += 1

print(f"✓ {edges_added} 엣지 추가 완료 ({edges_skipped} 건 스킵)")

# 가중치 분포 확인
weights = [d['weight'] for u, v, d in G.edges(data=True)]
print(f"\n엣지 가중치 통계:")
print(f"  - 평균: {np.mean(weights):.2f}")
print(f"  - 중앙값: {np.median(weights):.2f}")
print(f"  - 최소/최대: {min(weights):.0f} / {max(weights):.0f}")
print(f"  - 75% 백분위: {np.percentile(weights, 75):.0f}")

## Cell 12: 노드와 엣지에 원본 텍스트 추가

In [ ]:
print("=== 원본 텍스트 연결 중... ===")

# 노드의 원본 텍스트 추가
nodes_with_text = 0
for node in G.nodes():
    texts = []
    for tu_id in G.nodes[node]['text_unit_ids']:
        if tu_id in text_unit_lookup:
            texts.append({
                'id': tu_id,
                'text': text_unit_lookup[tu_id],
                'document': document_lookup.get(text_unit_to_doc.get(tu_id, ''), {}).get('title', 'Unknown')
            })
    
    if texts:
        G.nodes[node]['original_texts'] = texts
        nodes_with_text += 1

print(f"✓ {nodes_with_text} 노드에 원본 텍스트 연결 완료")

# 엣지의 원본 텍스트 추가
edges_with_text = 0
for u, v, key, data in G.edges(keys=True, data=True):
    texts = []
    for tu_id in data['text_unit_ids']:
        if tu_id in text_unit_lookup:
            texts.append(text_unit_lookup[tu_id])
    
    if texts:
        data['original_texts'] = texts
        edges_with_text += 1

print(f"✓ {edges_with_text} 엣지에 원본 텍스트 연결 완료")

## Cell 13: 그래프 기본 분석

In [ ]:
print("=== NetworkX 그래프 분석 ===")
print(f"\n1. 기본 통계:")
print(f"   - 노드 수: {G.number_of_nodes():,}")
print(f"   - 엣지 수: {G.number_of_edges():,}")
print(f"   - 평균 차수: {sum(dict(G.degree()).values()) / G.number_of_nodes():.2f}")

# 연결 요소 분석
print(f"\n2. 연결성 분석:")
print(f"   - 약한 연결 요소 수: {nx.number_weakly_connected_components(G)}")
print(f"   - 강한 연결 요소 수: {nx.number_strongly_connected_components(G)}")

# 최대 연결 요소 크기
largest_wcc = max(nx.weakly_connected_components(G), key=len)
print(f"   - 최대 약한 연결 요소 크기: {len(largest_wcc)} ({len(largest_wcc)/G.number_of_nodes()*100:.1f}%)")

# 중심성 계산 (샘플)
print(f"\n3. 중심성 분석 (계산 중...):")
degree_centrality = nx.degree_centrality(G)
pagerank = nx.pagerank(G, alpha=0.85, max_iter=100)

# 샘플링으로 betweenness 계산 (전체는 너무 오래 걸림)
sampled_nodes = list(G.nodes())[:100] if G.number_of_nodes() > 100 else list(G.nodes())
betweenness_centrality = nx.betweenness_centrality_subset(
    G, sampled_nodes, sampled_nodes, normalized=True
)

print("   ✓ 중심성 계산 완료")

# 가장 중심적인 노드들
print(f"\n4. Top 10 중심 노드 (PageRank 기준):")
for node, score in sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:10]:
    degree = G.degree(node)
    node_type = G.nodes[node]['type']
    print(f"   - {node:30} | PR: {score:.5f} | Degree: {degree:3} | Type: {node_type}")

# 경로 분석 샘플
print(f"\n5. Multi-hop 경로 샘플 찾기:")
sample_paths = []
nodes = list(G.nodes())
attempts = 0
max_attempts = 100

while len(sample_paths) < 5 and attempts < max_attempts:
    source, target = np.random.choice(nodes, 2, replace=False)
    attempts += 1
    
    try:
        path = nx.shortest_path(G, source, target)
        if 3 <= len(path) <= 5:  # 2-4 hop paths
            sample_paths.append(path)
            print(f"   - {' → '.join(path[:3])}... ({len(path)} 노드)")
    except nx.NetworkXNoPath:
        pass

print(f"\n✓ {len(sample_paths)} 개의 multi-hop 경로 샘플 발견")

## Cell 14: QA 생성용 특수 인덱스

In [ ]:
print("=== QA 생성을 위한 보조 구조 생성 중... ===")

# 1. 커뮤니티별 노드 그룹
nodes_by_community = defaultdict(lambda: defaultdict(list))
for node in G.nodes():
    for level, comm_info in G.nodes[node]['communities'].items():
        nodes_by_community[level][comm_info['id']].append(node)

print(f"\n1. 커뮤니티별 노드 그룹핑 완료")
for level in sorted(nodes_by_community.keys()):
    comm_count = len(nodes_by_community[level])
    avg_size = np.mean([len(nodes) for nodes in nodes_by_community[level].values()])
    print(f"   - Level {level}: {comm_count} 커뮤니티, 평균 {avg_size:.1f} 노드")

# 2. text_unit별 노드/엣지 그룹
nodes_by_text_unit = defaultdict(set)
edges_by_text_unit = defaultdict(set)

for node in G.nodes():
    for tu_id in G.nodes[node]['text_unit_ids']:
        nodes_by_text_unit[tu_id].add(node)

for u, v, data in G.edges(data=True):
    for tu_id in data['text_unit_ids']:
        edges_by_text_unit[tu_id].add((u, v))

print(f"\n2. Text unit별 그룹핑 완료")
print(f"   - {len(nodes_by_text_unit)} text units에 노드 분포")
print(f"   - {len(edges_by_text_unit)} text units에 엣지 분포")

# 3. 고가중치 경로 (인과관계 후보)
weight_threshold = np.percentile([d['weight'] for u, v, d in G.edges(data=True)], 75)
high_weight_edges = [(u, v, d) for u, v, d in G.edges(data=True) 
                     if d['weight'] > weight_threshold]

print(f"\n3. 고가중치 엣지 추출")
print(f"   - 임계값 (75%): {weight_threshold:.0f}")
print(f"   - 고가중치 엣지 수: {len(high_weight_edges)}")

# 인과관계 키워드 포함 엣지 찾기
causal_keywords = ['때문', '인해', '영향', '결과', '원인', '따라', '으로써', '위해']
causal_edges = []
for u, v, d in G.edges(data=True):
    if any(keyword in d['description'] for keyword in causal_keywords):
        causal_edges.append((u, v, d))

print(f"   - 인과관계 키워드 포함 엣지: {len(causal_edges)}")

# 4. 암시적 관계 후보 (같은 커뮤니티, 직접 연결 X)
print(f"\n4. 암시적 관계 후보 찾기...")
implicit_candidates = []

for level in range(min(3, len(nodes_by_community))):  # 상위 3개 레벨만
    for comm_id, nodes in nodes_by_community[level].items():
        if len(nodes) > 100:  # 너무 큰 커뮤니티는 샘플링
            nodes = np.random.choice(nodes, 100, replace=False)
        
        for i, node1 in enumerate(nodes):
            for node2 in nodes[i+1:min(i+6, len(nodes))]:  # 각 노드당 최대 5개만 체크
                if not G.has_edge(node1, node2) and not G.has_edge(node2, node1):
                    # 공통 이웃 확인
                    common_neighbors = set(G.neighbors(node1)) & set(G.neighbors(node2))
                    if common_neighbors:  # 공통 이웃이 있으면 암시적 관계 가능성
                        implicit_candidates.append({
                            'node1': node1,
                            'node2': node2,
                            'level': level,
                            'community': comm_id,
                            'common_neighbors': len(common_neighbors)
                        })

# 공통 이웃 수가 많은 순으로 정렬하고 상위 1000개만 유지
implicit_candidates = sorted(implicit_candidates, 
                           key=lambda x: x['common_neighbors'], 
                           reverse=True)[:1000]

print(f"✓ {len(implicit_candidates)} 개의 암시적 관계 후보 발견")

## Cell 15: 그래프 시각화 샘플

In [ ]:
print("=== 그래프 시각화 생성 중... ===")

try:
    from pyvis.network import Network
    import random
    
    # 가장 중심적인 노드 선택
    center_node = max(pagerank, key=pagerank.get)
    print(f"중심 노드: {center_node}")
    
    # 서브그래프 노드 선택
    subgraph_nodes = set([center_node])
    
    # 1차 이웃 추가
    for neighbor in list(G.neighbors(center_node))[:10]:
        subgraph_nodes.add(neighbor)
        
        # 2차 이웃 일부 추가
        for n2 in list(G.neighbors(neighbor))[:2]:
            subgraph_nodes.add(n2)
    
    print(f"서브그래프 노드 수: {len(subgraph_nodes)}")
    
    # PyVis 네트워크 생성
    net = Network(height='750px', width='100%', directed=True, notebook=True)
    net.barnes_hut(gravity=-80000, central_gravity=0.3, spring_length=250)
    
    # 서브그래프 추출
    subgraph = G.subgraph(subgraph_nodes)
    
    # 노드 추가
    for node in subgraph.nodes():
        node_data = G.nodes[node]
        
        # 노드 색상 결정
        if node_data['type'] == '작물':
            color = '#FF6B6B'
        elif node_data['type'] == '질병':
            color = '#4ECDC4'
        elif node_data['type'] == '조직':
            color = '#45B7D1'
        else:
            color = '#96CEB4'
        
        # 노드 크기는 degree에 비례
        size = min(10 + G.degree(node) * 2, 50)
        
        net.add_node(
            node, 
            label=node,
            title=f"{node}\\nType: {node_data['type']}\\nDegree: {G.degree(node)}",
            color=color,
            size=size
        )
    
    # 엣지 추가
    for u, v, data in subgraph.edges(data=True):
        net.add_edge(
            u, v, 
            title=f"Weight: {data['weight']:.0f}",
            width=min(data['weight']/20, 5),
            arrows='to'
        )
    
    # HTML 저장
    net.save_graph(str(output_path / 'sample_subgraph.html'))
    print(f"✓ 시각화 저장 완료: {output_path / 'sample_subgraph.html'}")
    
except ImportError:
    print("⚠️  pyvis가 설치되지 않음. 시각화를 건너뜁니다.")
    print("   설치하려면: pip install pyvis")

## Cell 16: 그래프 및 보조 구조 저장

In [ ]:
print("=== 최종 데이터 저장 중... ===")

# 저장할 데이터 구성
graph_data = {
    'graph': G,
    'nodes_by_community': dict(nodes_by_community),
    'nodes_by_text_unit': dict(nodes_by_text_unit),
    'edges_by_text_unit': dict(edges_by_text_unit),
    'high_weight_edges': high_weight_edges[:500],  # 상위 500개만
    'causal_edges': causal_edges[:200],  # 상위 200개만
    'implicit_candidates': implicit_candidates[:1000],  # 상위 1000개만
    'centrality_measures': {
        'degree': degree_centrality,
        'betweenness': dict(betweenness_centrality),
        'pagerank': pagerank
    },
    'sample_paths': sample_paths
}

# pickle로 저장
pkl_file = output_path / 'networkx_graph_data.pkl'
with open(pkl_file, 'wb') as f:
    pickle.dump(graph_data, f)

print(f"✓ NetworkX 그래프 데이터 저장 완료: {pkl_file}")
print(f"  파일 크기: {pkl_file.stat().st_size / 1024 / 1024:.2f} MB")

# 저장된 데이터 요약
print(f"\n저장된 데이터 구조:")
for key, value in graph_data.items():
    if key == 'graph':
        print(f"  - {key}: NetworkX Graph ({value.number_of_nodes()} nodes, {value.number_of_edges()} edges)")
    elif isinstance(value, dict):
        print(f"  - {key}: {len(value)} items")
    elif isinstance(value, list):
        print(f"  - {key}: {len(value)} items")
    else:
        print(f"  - {key}: {type(value)}")

## Cell 17: Phase 2 완료 요약

In [ ]:
print("=== Phase 2 완료 요약 ===")
print(f"\n1. 그래프 구축 완료:")
print(f"   - 노드: {G.number_of_nodes():,}")
print(f"   - 엣지: {G.number_of_edges():,}")
print(f"   - 평균 차수: {sum(dict(G.degree()).values()) / G.number_of_nodes():.2f}")
print(f"   - 커뮤니티 레벨: {len(nodes_by_community)}")

print(f"\n2. QA 생성 준비 완료:")
print(f"   - Multi-hop 경로 샘플: {len(sample_paths)}")
print(f"   - 암시적 관계 후보: {len(implicit_candidates)}")
print(f"   - 고가중치 엣지: {len(high_weight_edges)}")
print(f"   - 인과관계 후보 엣지: {len(causal_edges)}")
print(f"   - Text unit별 그룹핑: {len(nodes_by_text_unit)} text units")

print(f"\n3. 가장 중요한 노드 Top 10 (PageRank 기준):")
for i, (node, score) in enumerate(sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:10], 1):
    node_type = G.nodes[node]['type']
    degree = G.degree(node)
    communities = len(G.nodes[node]['communities'])
    print(f"   {i:2}. {node:25} | PR: {score:.5f} | Deg: {degree:3} | Type: {node_type:10} | Comm: {communities}")

print(f"\n4. 저장된 파일:")
print(f"   - NetworkX 그래프: {output_path / 'networkx_graph_data.pkl'}")
print(f"   - GraphML: {output_path / 'knowledge_graph.graphml'}")
if (output_path / 'sample_subgraph.html').exists():
    print(f"   - 시각화: {output_path / 'sample_subgraph.html'}")

print(f"\n✓ Phase 2 완료! Phase 3에서 QA 데이터셋 생성을 시작할 수 있습니다.")
print(f"\n다음 단계:")
print("1. Phase 3: QA 유형별 패턴 추출")
print("2. Phase 4: 질문 생성")
print("3. Phase 5: Ground Truth 생성")
print("4. Phase 6: 최종 데이터셋 구성")

# Phase 3: QA 유형별 패턴 추출

이 섹션에서는 GraphRAG의 장점을 보여줄 수 있는 5가지 유형의 QA 패턴을 추출합니다.

## Cell 18: Phase 2 데이터 로드 및 유틸리티 함수

In [ ]:
print("=== Phase 3 시작: QA 패턴 추출 ===")
print("\n1. Phase 2 데이터 로드 중...")

# NetworkX 그래프 데이터 로드
with open(output_path / 'networkx_graph_data.pkl', 'rb') as f:
    graph_data = pickle.load(f)

# 그래프 및 보조 구조 언패킹
G = graph_data['graph']
nodes_by_community = graph_data['nodes_by_community']
nodes_by_text_unit = graph_data['nodes_by_text_unit']
edges_by_text_unit = graph_data['edges_by_text_unit']
high_weight_edges = graph_data['high_weight_edges']
causal_edges = graph_data['causal_edges']
implicit_candidates = graph_data['implicit_candidates']
centrality_measures = graph_data['centrality_measures']
sample_paths = graph_data['sample_paths']

print(f"✓ NetworkX 그래프 로드 완료: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Phase 1 데이터 로드 (텍스트 접근용)
with open(output_path / 'processed_graphrag_data.pkl', 'rb') as f:
    processed_data = pickle.load(f)

text_unit_lookup = processed_data['text_unit_lookup']
entity_to_communities = processed_data['entity_to_communities']
community_reports_lookup = processed_data['community_reports_lookup']
document_lookup = processed_data['document_lookup']
text_unit_to_doc = processed_data['text_unit_to_doc']
entities_by_id = processed_data['entities_by_id']
entities_by_title = processed_data['entities_by_title']

print("✓ 텍스트 및 메타데이터 로드 완료")

print("\n2. 유틸리티 함수 정의 중...")

def get_path_context(path: List[str]) -> Dict:
    """경로상의 모든 노드와 엣지 정보 수집"""
    context = {
        'path': path,
        'nodes': {},
        'edges': [],
        'texts': set(),
        'documents': set()
    }
    
    # 노드 정보 수집
    for node in path:
        node_data = G.nodes[node]
        context['nodes'][node] = {
            'type': node_data['type'],
            'description': node_data['description'],
            'communities': node_data.get('communities', {})
        }
        # 원본 텍스트 수집
        for text_info in node_data.get('original_texts', []):
            context['texts'].add(text_info['text'])
            context['documents'].add(text_info.get('document', 'Unknown'))
    
    # 엣지 정보 수집
    for i in range(len(path)-1):
        if G.has_edge(path[i], path[i+1]):
            edge_data = G.get_edge_data(path[i], path[i+1])
            # MultiDiGraph인 경우 첫 번째 엣지 사용
            if isinstance(edge_data, dict) and 0 in edge_data:
                edge_data = edge_data[0]
            context['edges'].append({
                'source': path[i],
                'target': path[i+1],
                'description': edge_data.get('description', ''),
                'weight': edge_data.get('weight', 0)
            })
            # 엣지 텍스트도 수집
            for text in edge_data.get('original_texts', []):
                context['texts'].add(text)
    
    return context

def get_entity_full_context(entity_name: str) -> Dict:
    """엔티티의 모든 관련 정보 수집"""
    if entity_name not in G:
        return None
    
    node_data = G.nodes[entity_name]
    entity_info = entities_by_title.get(entity_name, {})
    
    context = {
        'name': entity_name,
        'type': node_data['type'],
        'description': node_data['description'],
        'degree': G.degree(entity_name),
        'communities': node_data.get('communities', {}),
        'neighbors': [],
        'texts': []
    }
    
    # 이웃 노드 정보
    for neighbor in G.neighbors(entity_name):
        edge_data = G.get_edge_data(entity_name, neighbor)
        if isinstance(edge_data, dict) and 0 in edge_data:
            edge_data = edge_data[0]
        context['neighbors'].append({
            'node': neighbor,
            'relation': edge_data.get('description', ''),
            'weight': edge_data.get('weight', 0)
        })
    
    # 원본 텍스트
    for text_info in node_data.get('original_texts', []):
        context['texts'].append(text_info['text'])
    
    return context

def get_community_context(community_id: int, level: int) -> Dict:
    """커뮤니티 정보와 주요 멤버 수집"""
    key = (community_id, level)
    report = community_reports_lookup.get(key, {})
    members = nodes_by_community.get(level, {}).get(community_id, [])
    
    context = {
        'id': community_id,
        'level': level,
        'title': report.get('title', ''),
        'summary': report.get('summary', ''),
        'findings': report.get('findings', []),
        'members': members[:20],  # 상위 20개만
        'member_count': len(members),
        'key_relationships': []
    }
    
    # 커뮤니티 내 주요 관계 찾기
    for i, node1 in enumerate(members[:10]):
        for node2 in members[i+1:10]:
            if G.has_edge(node1, node2):
                edge_data = G.get_edge_data(node1, node2)
                if isinstance(edge_data, dict) and 0 in edge_data:
                    edge_data = edge_data[0]
                context['key_relationships'].append({
                    'source': node1,
                    'target': node2,
                    'description': edge_data.get('description', '')
                })
    
    return context

def validate_qa_pattern(pattern: Dict) -> bool:
    """QA 패턴이 유효한지 검증"""
    # 기본 검증 기준
    if not pattern.get('entities') or not pattern.get('question_type'):
        return False
    
    # 엔티티가 모두 그래프에 존재하는지
    for entity in pattern.get('entities', []):
        if entity not in G:
            return False
    
    # 텍스트 근거가 있는지
    if not pattern.get('source_texts'):
        return False
    
    return True

print("✓ 유틸리티 함수 정의 완료")
print("\n사용 가능한 함수:")
print("  - get_path_context(path): 경로 정보 수집")
print("  - get_entity_full_context(entity): 엔티티 전체 정보")
print("  - get_community_context(id, level): 커뮤니티 정보")
print("  - validate_qa_pattern(pattern): 패턴 유효성 검증")

# 데이터 요약
print("\n3. 로드된 데이터 요약:")
print(f"  - 고가중치 엣지: {len(high_weight_edges)}")
print(f"  - 인과관계 후보: {len(causal_edges)}")
print(f"  - 암시적 관계 후보: {len(implicit_candidates)}")
print(f"  - 샘플 경로: {len(sample_paths)}")
print(f"  - 커뮤니티 레벨: {len(nodes_by_community)}")
print(f"  - Text units: {len(nodes_by_text_unit)}")

print("\n✓ Phase 3 준비 완료! 다음 단계에서 각 QA 유형별 패턴을 추출합니다.")

## Cell 19: Multi-hop Reasoning 패턴 추출

2-hop과 3-hop 경로를 추출합니다. (비율: 2-hop 75%, 3-hop 25%)

In [ ]:
print("=== Multi-hop Reasoning 패턴 추출 ===")

# Multi-hop 패턴 저장
multi_hop_patterns = []

# 1. 중요 노드 선정 (PageRank 상위)
pagerank = centrality_measures['pagerank']
top_nodes = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:50]
important_nodes = [node for node, _ in top_nodes]

print(f"상위 {len(important_nodes)}개 중요 노드 선정")

# 2. 2-hop 패턴 추출 (목표: 75개)
print("\n1. 2-hop 패턴 추출 중...")
two_hop_patterns = []

for source in important_nodes[:30]:  # 상위 30개 노드에서 시작
    # source의 이웃들
    for mid in list(G.neighbors(source))[:5]:  # 각 노드당 최대 5개 이웃
        # mid의 이웃들 (source 제외)
        for target in list(G.neighbors(mid))[:3]:  # 각 중간노드당 최대 3개
            if target != source and target in G:
                # 경로 컨텍스트 수집
                path = [source, mid, target]
                context = get_path_context(path)
                
                # 유효성 검증
                if len(context['texts']) >= 2 and len(context['edges']) == 2:
                    pattern = {
                        'question_type': 'multi_hop',
                        'hop_count': 2,
                        'path': path,
                        'entities': path,
                        'relationships': context['edges'],
                        'source_texts': list(context['texts'])[:3],  # 최대 3개 텍스트
                        'difficulty': 'easy',
                        'pattern_template': f"{source}와(과) {target}의 관계는 무엇인가? ({mid}를 통해)",
                        'node_types': [G.nodes[n]['type'] for n in path]
                    }
                    
                    # 중복 체크 (동일한 시작-끝 노드 쌍)
                    if not any(p['entities'][0] == source and p['entities'][-1] == target 
                              for p in two_hop_patterns):
                        two_hop_patterns.append(pattern)
                        
                if len(two_hop_patterns) >= 75:
                    break
        if len(two_hop_patterns) >= 75:
            break

print(f"✓ 2-hop 패턴 {len(two_hop_patterns)}개 추출")

# 3. 3-hop 패턴 추출 (목표: 25개)
print("\n2. 3-hop 패턴 추출 중...")
three_hop_patterns = []

for source in important_nodes[:20]:  # 상위 20개 노드에서 시작
    # BFS로 3-hop 경로 찾기
    try:
        # source에서 도달 가능한 노드들
        reachable = nx.single_source_shortest_path_length(G, source, cutoff=3)
        
        # 정확히 3-hop 거리의 노드들
        three_hop_targets = [node for node, dist in reachable.items() if dist == 3]
        
        for target in three_hop_targets[:3]:  # 각 source당 최대 3개
            try:
                # 최단 경로 찾기
                paths = list(nx.all_shortest_paths(G, source, target))
                if paths and len(paths[0]) == 4:  # 4개 노드 = 3-hop
                    path = paths[0]
                    context = get_path_context(path)
                    
                    # 유효성 검증
                    if len(context['texts']) >= 2 and len(context['edges']) == 3:
                        # 엔티티 타입이 다양한지 체크
                        node_types = [G.nodes[n]['type'] for n in path]
                        unique_types = len(set(node_types))
                        
                        pattern = {
                            'question_type': 'multi_hop',
                            'hop_count': 3,
                            'path': path,
                            'entities': path,
                            'relationships': context['edges'],
                            'source_texts': list(context['texts'])[:4],  # 최대 4개 텍스트
                            'difficulty': 'medium',
                            'pattern_template': f"{source}이(가) {target}에 미치는 영향의 경로는?",
                            'node_types': node_types,
                            'type_diversity': unique_types
                        }
                        
                        # 중복 체크 및 다양성 확보
                        if (not any(p['entities'][0] == source and p['entities'][-1] == target 
                                   for p in three_hop_patterns) 
                            and unique_types >= 2):  # 최소 2개 이상 다른 타입
                            three_hop_patterns.append(pattern)
                            
            except nx.NetworkXNoPath:
                continue
                
            if len(three_hop_patterns) >= 25:
                break
                
    except Exception as e:
        continue
        
    if len(three_hop_patterns) >= 25:
        break

print(f"✓ 3-hop 패턴 {len(three_hop_patterns)}개 추출")

# 4. 패턴 통합 및 정리
multi_hop_patterns = two_hop_patterns + three_hop_patterns
print(f"\n3. 총 {len(multi_hop_patterns)}개 multi-hop 패턴 추출")

# 5. 패턴 분석
print("\n4. 패턴 분석:")
print(f"  - 2-hop: {len(two_hop_patterns)} ({len(two_hop_patterns)/len(multi_hop_patterns)*100:.1f}%)")
print(f"  - 3-hop: {len(three_hop_patterns)} ({len(three_hop_patterns)/len(multi_hop_patterns)*100:.1f}%)")

# 엔티티 타입 분포
all_types = []
for pattern in multi_hop_patterns:
    all_types.extend(pattern['node_types'])
type_counts = pd.Series(all_types).value_counts().head(10)
print("\n  엔티티 타입 분포:")
for t, c in type_counts.items():
    print(f"    - {t}: {c}")

# 샘플 패턴 출력
print("\n5. 샘플 패턴:")
for i, pattern in enumerate(multi_hop_patterns[:3]):
    print(f"\n  패턴 {i+1} ({pattern['hop_count']}-hop):")
    print(f"    경로: {' → '.join(pattern['path'])}")
    print(f"    타입: {' → '.join(pattern['node_types'])}")
    print(f"    템플릿: {pattern['pattern_template']}")
    if pattern['relationships']:
        print(f"    관계 설명: {pattern['relationships'][0]['description'][:50]}...")

print(f"\n✓ Multi-hop 패턴 추출 완료!")
print(f"다음 단계: Community synthesis 패턴 추출")

### Cell 20: 2-hop 패턴 상세 분석

추출된 2-hop 패턴들이 실제로 어떤 노드를 거쳐가는지 상세히 분석합니다.

In [ ]:
print("=== 2-hop 패턴 상세 분석 ===\n")

# 2-hop 패턴의 구조적 특성 분석
print("1. 2-hop 패턴 구조 분석:")
print(f"   총 {len(two_hop_patterns)}개의 2-hop 패턴\n")

# 패턴 타입별 분류
pattern_types = defaultdict(list)
for pattern in two_hop_patterns:
    # 경로: A → B → C
    a_type = pattern['node_types'][0]
    b_type = pattern['node_types'][1]
    c_type = pattern['node_types'][2]
    
    pattern_key = f"{a_type} → {b_type} → {c_type}"
    pattern_types[pattern_key].append(pattern)

# 가장 많이 나타나는 패턴 타입
print("2. 주요 멀티홉 템플릿 (노드 타입 기준):")
sorted_patterns = sorted(pattern_types.items(), key=lambda x: len(x[1]), reverse=True)[:10]

template_analysis = []
for i, (pattern_template, patterns) in enumerate(sorted_patterns, 1):
    print(f"\n   템플릿 {i}: {pattern_template}")
    print(f"   출현 빈도: {len(patterns)}회")
    
    # 샘플 경로 보여주기
    sample = patterns[0]
    path = sample['path']
    print(f"   예시 경로: {path[0]} → {path[1]} → {path[2]}")
    
    # 관계 설명
    rel_descriptions = []
    if sample['relationships']:
        rel1 = sample['relationships'][0]['description'][:40]
        rel2 = sample['relationships'][1]['description'][:40]
        print(f"   관계1: {path[0]} → {path[1]}: {rel1}...")
        print(f"   관계2: {path[1]} → {path[2]}: {rel2}...")
        rel_descriptions = [sample['relationships'][0]['description'], sample['relationships'][1]['description']]
    
    template_analysis.append({
        'template': pattern_template,
        'frequency': len(patterns),
        'sample_path': path,
        'sample_relations': rel_descriptions
    })

# 중간 노드 역할 분석
print("\n\n3. 중간 노드(Bridge Node) 분석:")
bridge_nodes = defaultdict(int)
bridge_node_contexts = defaultdict(list)

for pattern in two_hop_patterns:
    bridge = pattern['path'][1]
    bridge_nodes[bridge] += 1
    bridge_node_contexts[bridge].append({
        'source': pattern['path'][0],
        'target': pattern['path'][2],
        'source_type': pattern['node_types'][0],
        'target_type': pattern['node_types'][2]
    })

# 가장 자주 브릿지 역할을 하는 노드들
print("   주요 브릿지 노드 Top 10:")
sorted_bridges = sorted(bridge_nodes.items(), key=lambda x: x[1], reverse=True)[:10]

bridge_analysis = []
for node, count in sorted_bridges:
    node_type = G.nodes[node]['type']
    degree = G.degree(node)
    
    # 연결하는 노드 타입의 다양성
    contexts = bridge_node_contexts[node]
    unique_source_types = len(set(c['source_type'] for c in contexts))
    unique_target_types = len(set(c['target_type'] for c in contexts))
    
    print(f"   - {node} (타입: {node_type}, 연결도: {degree})")
    print(f"     브릿지 역할: {count}회")
    print(f"     연결 다양성: {unique_source_types}개 소스 타입 × {unique_target_types}개 타겟 타입")
    
    bridge_analysis.append({
        'node': node,
        'type': node_type,
        'degree': degree,
        'bridge_count': count,
        'source_type_diversity': unique_source_types,
        'target_type_diversity': unique_target_types,
        'contexts': contexts[:5]  # 상위 5개만 저장
    })

# 의미론적 패턴 분석
print("\n\n4. 의미론적 연결 패턴:")

semantic_patterns = {
    '인과관계': [],
    '속성관계': [],
    '계층관계': [],
    '연관관계': []
}

# 관계 설명에 기반한 패턴 분류
causal_keywords = ['때문', '인해', '영향', '결과', '원인']
attribute_keywords = ['특징', '속성', '성질', '포함', '구성']
hierarchy_keywords = ['종류', '분류', '속한', '하위', '상위']

for pattern in two_hop_patterns[:30]:  # 상위 30개만 분석
    rel_descriptions = [r['description'] for r in pattern['relationships']]
    all_desc = ' '.join(rel_descriptions)
    
    if any(k in all_desc for k in causal_keywords):
        semantic_patterns['인과관계'].append(pattern)
    elif any(k in all_desc for k in attribute_keywords):
        semantic_patterns['속성관계'].append(pattern)
    elif any(k in all_desc for k in hierarchy_keywords):
        semantic_patterns['계층관계'].append(pattern)
    else:
        semantic_patterns['연관관계'].append(pattern)

semantic_analysis = {}
print("   의미론적 패턴 분포:")
for pattern_type, patterns in semantic_patterns.items():
    if patterns:
        print(f"\n   {pattern_type}: {len(patterns)}개")
        # 샘플 출력
        sample = patterns[0]
        path = sample['path']
        print(f"     예시: {path[0]} → {path[1]} → {path[2]}")
        print(f"     설명: {sample['relationships'][0]['description'][:50]}...")
        
        semantic_analysis[pattern_type] = {
            'count': len(patterns),
            'sample_paths': [p['path'] for p in patterns[:3]]
        }

# 질문 생성 가능성 평가
print("\n\n5. 질문 생성 적합도 평가:")

high_quality_patterns = []
quality_scores = []
for pattern in two_hop_patterns:
    score = 0
    
    # 텍스트 증거 충분성
    if len(pattern['source_texts']) >= 3:
        score += 2
    elif len(pattern['source_texts']) >= 2:
        score += 1
    
    # 노드 타입 다양성
    unique_types = len(set(pattern['node_types']))
    score += unique_types
    
    # 관계 설명 품질
    rel_desc_length = sum(len(r['description']) for r in pattern['relationships'])
    if rel_desc_length > 50:
        score += 2
    elif rel_desc_length > 20:
        score += 1
    
    # 중요도 (PageRank 기준)
    if pattern['path'][0] in [n for n, _ in top_nodes[:10]]:
        score += 1
    
    pattern['quality_score'] = score
    quality_scores.append(score)
    if score >= 5:
        high_quality_patterns.append(pattern)

print(f"   고품질 패턴: {len(high_quality_patterns)}개 / {len(two_hop_patterns)}개")
print(f"   평균 품질 점수: {sum(p.get('quality_score', 0) for p in two_hop_patterns) / len(two_hop_patterns):.2f}")

# 최고 품질 패턴 예시
print("\n   최고 품질 2-hop 패턴 예시:")
top_quality = sorted(two_hop_patterns, key=lambda x: x.get('quality_score', 0), reverse=True)[:3]

top_quality_examples = []
for i, pattern in enumerate(top_quality, 1):
    print(f"\n   예시 {i} (점수: {pattern.get('quality_score', 0)})")
    path = pattern['path']
    print(f"   경로: {path[0]} → {path[1]} → {path[2]}")
    print(f"   타입: {' → '.join(pattern['node_types'])}")
    print(f"   관계:")
    for j, rel in enumerate(pattern['relationships']):
        print(f"     {j+1}. {rel['source']} → {rel['target']}: {rel['description'][:60]}...")
    print(f"   텍스트 증거: {len(pattern['source_texts'])}개")
    
    top_quality_examples.append({
        'path': path,
        'node_types': pattern['node_types'],
        'relationships': pattern['relationships'],
        'quality_score': pattern.get('quality_score', 0),
        'text_count': len(pattern['source_texts'])
    })

# 분석 결과 저장
analysis_results = {
    'pattern_count': len(two_hop_patterns),
    'template_analysis': template_analysis[:10],
    'bridge_analysis': bridge_analysis,
    'semantic_analysis': semantic_analysis,
    'quality_distribution': {
        'high_quality_count': len(high_quality_patterns),
        'average_score': sum(quality_scores) / len(quality_scores),
        'score_distribution': pd.Series(quality_scores).value_counts().to_dict()
    },
    'top_quality_examples': top_quality_examples
}

# JSON으로 저장
import json
output_dir = output_path / 'multi_hop'
output_dir.mkdir(exist_ok=True)

with open(output_dir / '2hop_pattern_analysis.json', 'w', encoding='utf-8') as f:
    json.dump(analysis_results, f, ensure_ascii=False, indent=2)

# 패턴 자체도 저장 (추후 질문 생성에 사용)
with open(output_dir / '2hop_patterns.json', 'w', encoding='utf-8') as f:
    # JSON 직렬화를 위해 set을 list로 변환
    patterns_to_save = []
    for p in two_hop_patterns:
        pattern_copy = p.copy()
        pattern_copy['source_texts'] = list(pattern_copy['source_texts'])[:5]  # 상위 5개만
        patterns_to_save.append(pattern_copy)
    json.dump(patterns_to_save, f, ensure_ascii=False, indent=2)

print(f"\n✓ 2-hop 패턴 분석 결과 저장 완료:")
print(f"   - {output_dir / '2hop_pattern_analysis.json'}")
print(f"   - {output_dir / '2hop_patterns.json'}")
print("\n✓ 2-hop 패턴 상세 분석 완료!")

### Cell 21: 3-hop 패턴 상세 분석

추출된 3-hop 패턴들이 실제로 어떤 노드를 거쳐가는지 상세히 분석합니다.

In [ ]:
print("=== 3-hop 패턴 상세 분석 ===\n")

# 3-hop 패턴의 구조적 특성 분석
print("1. 3-hop 패턴 구조 분석:")
print(f"   총 {len(three_hop_patterns)}개의 3-hop 패턴\n")

# 패턴 타입별 분류
pattern_types_3hop = defaultdict(list)
for pattern in three_hop_patterns:
    # 경로: A → B → C → D
    types = pattern['node_types']
    pattern_key = ' → '.join(types)
    pattern_types_3hop[pattern_key].append(pattern)

# 가장 많이 나타나는 패턴 타입
print("2. 주요 3-hop 템플릿 (노드 타입 기준):")
sorted_patterns_3hop = sorted(pattern_types_3hop.items(), key=lambda x: len(x[1]), reverse=True)[:10]

template_analysis_3hop = []
for i, (pattern_template, patterns) in enumerate(sorted_patterns_3hop, 1):
    print(f"\n   템플릿 {i}: {pattern_template}")
    print(f"   출현 빈도: {len(patterns)}회")
    print(f"   타입 다양성: {patterns[0]['type_diversity']}개 고유 타입")
    
    # 샘플 경로 보여주기
    sample = patterns[0]
    path = sample['path']
    print(f"   예시 경로: {' → '.join(path)}")
    
    template_analysis_3hop.append({
        'template': pattern_template,
        'frequency': len(patterns),
        'type_diversity': patterns[0]['type_diversity'],
        'sample_path': path,
        'sample_relations': [r['description'] for r in sample['relationships']]
    })

# 3-hop 경로의 중간 노드 역할 분석
print("\n\n3. 3-hop 경로의 중간 노드 체인 분석:")

# 첫 번째와 두 번째 중간 노드 분석
first_bridges = defaultdict(int)
second_bridges = defaultdict(int)
bridge_pairs = defaultdict(int)

for pattern in three_hop_patterns:
    path = pattern['path']
    first_bridge = path[1]
    second_bridge = path[2]
    
    first_bridges[first_bridge] += 1
    second_bridges[second_bridge] += 1
    bridge_pairs[(first_bridge, second_bridge)] += 1

bridge_chain_analysis = {
    'first_bridges': [],
    'second_bridges': [],
    'bridge_pairs': []
}

print("   3.1. 첫 번째 브릿지 노드 (A → B):")
for node, count in sorted(first_bridges.items(), key=lambda x: x[1], reverse=True)[:5]:
    node_type = G.nodes[node]['type']
    print(f"       - {node} ({node_type}): {count}회")
    bridge_chain_analysis['first_bridges'].append({
        'node': node,
        'type': node_type,
        'count': count
    })

print("\n   3.2. 두 번째 브릿지 노드 (B → C):")
for node, count in sorted(second_bridges.items(), key=lambda x: x[1], reverse=True)[:5]:
    node_type = G.nodes[node]['type']
    print(f"       - {node} ({node_type}): {count}회")
    bridge_chain_analysis['second_bridges'].append({
        'node': node,
        'type': node_type,
        'count': count
    })

print("\n   3.3. 자주 나타나는 브릿지 페어:")
for (node1, node2), count in sorted(bridge_pairs.items(), key=lambda x: x[1], reverse=True)[:5]:
    type1 = G.nodes[node1]['type']
    type2 = G.nodes[node2]['type']
    print(f"       - {node1}({type1}) → {node2}({type2}): {count}회")
    bridge_chain_analysis['bridge_pairs'].append({
        'pair': [node1, node2],
        'types': [type1, type2],
        'count': count
    })

# 경로 복잡도 분석
print("\n\n4. 3-hop 경로 복잡도 분석:")

complexity_scores = []
complexity_details = []
for pattern in three_hop_patterns:
    path = pattern['path']
    
    # 복잡도 점수 계산
    complexity = {
        'type_diversity': pattern['type_diversity'],
        'total_degree': sum(G.degree(node) for node in path),
        'min_degree': min(G.degree(node) for node in path),
        'max_degree': max(G.degree(node) for node in path),
        'relation_weights': [r['weight'] for r in pattern['relationships']],
        'text_evidence': len(pattern['source_texts'])
    }
    
    # 종합 복잡도 점수
    score = (
        complexity['type_diversity'] * 2 +  # 타입 다양성 중요
        min(complexity['text_evidence'], 4) +  # 텍스트 증거
        (1 if complexity['min_degree'] > 2 else 0) +  # 모든 노드가 충분히 연결됨
        (1 if max(complexity['relation_weights']) > 50 else 0)  # 강한 관계 포함
    )
    
    pattern['complexity_score'] = score
    complexity_scores.append(score)
    complexity_details.append({
        'path': path,
        'score': score,
        'details': complexity
    })

avg_complexity = sum(complexity_scores) / len(complexity_scores)
print(f"   평균 복잡도 점수: {avg_complexity:.2f}")
print(f"   고복잡도 패턴 (점수 >= 5): {len([s for s in complexity_scores if s >= 5])}개")

# 의미론적 체인 분석
print("\n\n5. 의미론적 연결 체인 분석:")

chain_types = {
    '순차적 인과': [],  # A → B → C → D (각 단계가 다음 단계의 원인)
    '분기 후 수렴': [],  # A가 여러 경로로 D에 영향
    '계층적 전파': [],  # 상위 → 중간 → 하위 개념
    '복합 연관': []  # 기타 복잡한 관계
}

for pattern in three_hop_patterns:
    # 관계들의 의미론적 특성 분석
    relations = pattern['relationships']
    
    # 인과관계 키워드 체크
    causal_count = sum(1 for r in relations if any(k in r['description'] for k in ['때문', '인해', '영향', '결과']))
    hierarchy_count = sum(1 for r in relations if any(k in r['description'] for k in ['종류', '분류', '속한']))
    
    if causal_count >= 2:
        chain_types['순차적 인과'].append(pattern)
    elif hierarchy_count >= 2:
        chain_types['계층적 전파'].append(pattern)
    elif pattern['type_diversity'] >= 3:
        chain_types['분기 후 수렴'].append(pattern)
    else:
        chain_types['복합 연관'].append(pattern)

chain_analysis = {}
print("   의미론적 체인 유형 분포:")
for chain_type, patterns in chain_types.items():
    if patterns:
        print(f"\n   {chain_type}: {len(patterns)}개")
        # 샘플 출력
        if patterns:
            sample = patterns[0]
            path = sample['path']
            print(f"     예시: {' → '.join(path[:2])}...{path[-1]}")
        
        chain_analysis[chain_type] = {
            'count': len(patterns),
            'sample_paths': [p['path'] for p in patterns[:3]]
        }

# 최고 품질 3-hop 패턴
print("\n\n6. 최고 품질 3-hop 패턴 예시:")
top_3hop = sorted(three_hop_patterns, 
                  key=lambda x: x.get('complexity_score', 0) * 10 + len(x['source_texts']), 
                  reverse=True)[:3]

top_quality_3hop = []
for i, pattern in enumerate(top_3hop, 1):
    print(f"\n   예시 {i}:")
    path = pattern['path']
    print(f"   전체 경로: {path[0]} → {path[1]} → {path[2]} → {path[3]}")
    print(f"   노드 타입: {' → '.join(pattern['node_types'])}")
    print(f"   복잡도 점수: {pattern.get('complexity_score', 0)}")
    print(f"   타입 다양성: {pattern['type_diversity']}개")
    print(f"   텍스트 증거: {len(pattern['source_texts'])}개")
    
    print(f"   관계 체인:")
    relations_detail = []
    for j, rel in enumerate(pattern['relationships']):
        weight_indicator = "★" * min(int(rel['weight'] / 20), 5)
        print(f"     단계 {j+1}: {rel['description'][:50]}... {weight_indicator}")
        relations_detail.append({
            'step': j+1,
            'description': rel['description'],
            'weight': rel['weight']
        })
    
    top_quality_3hop.append({
        'path': path,
        'node_types': pattern['node_types'],
        'complexity_score': pattern.get('complexity_score', 0),
        'type_diversity': pattern['type_diversity'],
        'text_count': len(pattern['source_texts']),
        'relations': relations_detail
    })

# 분석 결과 저장
analysis_results_3hop = {
    'pattern_count': len(three_hop_patterns),
    'template_analysis': template_analysis_3hop[:10],
    'bridge_chain_analysis': bridge_chain_analysis,
    'complexity_analysis': {
        'average_score': avg_complexity,
        'high_complexity_count': len([s for s in complexity_scores if s >= 5]),
        'score_distribution': pd.Series(complexity_scores).value_counts().to_dict(),
        'top_complex_paths': sorted(complexity_details, key=lambda x: x['score'], reverse=True)[:5]
    },
    'chain_analysis': chain_analysis,
    'top_quality_examples': top_quality_3hop
}

# JSON으로 저장
output_dir = output_path / 'multi_hop'
output_dir.mkdir(exist_ok=True)

with open(output_dir / '3hop_pattern_analysis.json', 'w', encoding='utf-8') as f:
    json.dump(analysis_results_3hop, f, ensure_ascii=False, indent=2)

# 패턴 자체도 저장 (추후 질문 생성에 사용)
with open(output_dir / '3hop_patterns.json', 'w', encoding='utf-8') as f:
    # JSON 직렬화를 위해 set을 list로 변환
    patterns_to_save = []
    for p in three_hop_patterns:
        pattern_copy = p.copy()
        pattern_copy['source_texts'] = list(pattern_copy['source_texts'])[:5]  # 상위 5개만
        patterns_to_save.append(pattern_copy)
    json.dump(patterns_to_save, f, ensure_ascii=False, indent=2)

print(f"\n✓ 3-hop 패턴 분석 결과 저장 완료:")
print(f"   - {output_dir / '3hop_pattern_analysis.json'}")
print(f"   - {output_dir / '3hop_patterns.json'}")
print("\n✓ 3-hop 패턴 상세 분석 완료!")

## Cell 22: Community Synthesis 패턴 추출

커뮤니티 레벨의 요약 정보를 활용하여 여러 엔티티를 아우르는 통합적인 패턴을 추출합니다.

In [ ]:
print("=== Community Synthesis 패턴 추출 ===")

# Community synthesis 패턴 저장
community_synthesis_patterns = []

# 1. 각 레벨별 커뮤니티 분석
print("\n1. 커뮤니티 레벨별 분석:")
for level in sorted(nodes_by_community.keys()):
    communities = nodes_by_community[level]
    print(f"\n   Level {level}: {len(communities)} 커뮤니티")
    
    # 크기별 분포
    sizes = [len(nodes) for nodes in communities.values()]
    print(f"     - 평균 크기: {np.mean(sizes):.1f} 노드")
    print(f"     - 최대/최소: {max(sizes)} / {min(sizes)}")

# 2. 의미 있는 커뮤니티 선별 (중간 크기, 다양한 타입)
print("\n2. 패턴 추출을 위한 커뮤니티 선별...")

selected_communities = []
for level in sorted(nodes_by_community.keys())[:3]:  # 상위 3개 레벨만
    for comm_id, nodes in nodes_by_community[level].items():
        # 너무 크거나 작은 커뮤니티 제외
        if 5 <= len(nodes) <= 50:
            # 노드 타입 다양성 체크
            node_types = set()
            for node in nodes[:20]:  # 상위 20개만 체크
                if node in G:
                    node_types.add(G.nodes[node]['type'])
            
            # 리포트 존재 여부 확인
            report_key = (comm_id, level)
            report = community_reports_lookup.get(report_key, {})
            
            if len(node_types) >= 2 and report.get('summary'):  # 다양성 있고 요약이 있는 경우
                selected_communities.append({
                    'level': level,
                    'id': comm_id,
                    'size': len(nodes),
                    'node_types': node_types,
                    'report': report,
                    'nodes': nodes[:30]  # 최대 30개 노드만 저장
                })

print(f"   선별된 커뮤니티: {len(selected_communities)}개")

# 3. Community synthesis 패턴 생성
print("\n3. Community synthesis 패턴 생성 중...")

for comm_info in selected_communities[:40]:  # 최대 40개 커뮤니티
    level = comm_info['level']
    comm_id = comm_info['id']
    nodes = comm_info['nodes']
    report = comm_info['report']
    
    # 커뮤니티 내 주요 엔티티 선정 (degree 기준)
    node_degrees = [(node, G.degree(node)) for node in nodes if node in G]
    node_degrees.sort(key=lambda x: x[1], reverse=True)
    key_entities = [node for node, _ in node_degrees[:5]]  # 상위 5개
    
    # 커뮤니티 내 주요 관계 찾기
    internal_edges = []
    edge_count = 0
    for i, node1 in enumerate(nodes):
        for node2 in nodes[i+1:]:
            if G.has_edge(node1, node2):
                edge_data = G.get_edge_data(node1, node2)
                if isinstance(edge_data, dict) and 0 in edge_data:
                    edge_data = edge_data[0]
                internal_edges.append({
                    'source': node1,
                    'target': node2,
                    'description': edge_data.get('description', ''),
                    'weight': edge_data.get('weight', 0)
                })
                edge_count += 1
                if edge_count >= 10:  # 최대 10개 관계만
                    break
        if edge_count >= 10:
            break
    
    # 텍스트 증거 수집
    community_texts = set()
    for node in key_entities:
        if node in G:
            for text_info in G.nodes[node].get('original_texts', [])[:2]:
                community_texts.add(text_info['text'])
    
    if len(community_texts) >= 2:  # 충분한 텍스트 증거가 있는 경우
        pattern = {
            'question_type': 'community_synthesis',
            'level': level,
            'community_id': comm_id,
            'community_title': report.get('title', 'Community'),
            'community_summary': report.get('summary', ''),
            'entities': key_entities,
            'all_entities': nodes,
            'entity_count': len(nodes),
            'entity_types': list(comm_info['node_types']),
            'internal_relationships': internal_edges[:5],  # 주요 5개 관계
            'source_texts': list(community_texts)[:5],  # 최대 5개 텍스트
            'pattern_template': f"{report.get('title', 'Community')}에 속한 주요 요소들의 상호작용은?",
            'difficulty': 'medium' if level <= 1 else 'hard'
        }
        
        community_synthesis_patterns.append(pattern)

print(f"\n✓ {len(community_synthesis_patterns)}개 community synthesis 패턴 추출")

# 4. 패턴 통계
print("\n4. Community synthesis 패턴 통계:")

# 레벨별 분포
level_counts = defaultdict(int)
for pattern in community_synthesis_patterns:
    level_counts[pattern['level']] += 1

print("   레벨별 분포:")
for level in sorted(level_counts.keys()):
    count = level_counts[level]
    print(f"     - Level {level}: {count}개 ({count/len(community_synthesis_patterns)*100:.1f}%)")

# 커뮤니티 크기 분포
sizes = [p['entity_count'] for p in community_synthesis_patterns]
print(f"\n   커뮤니티 크기:")
print(f"     - 평균: {np.mean(sizes):.1f} 엔티티")
print(f"     - 범위: {min(sizes)} ~ {max(sizes)}")

# 엔티티 타입 다양성
type_diversities = [len(p['entity_types']) for p in community_synthesis_patterns]
print(f"\n   타입 다양성:")
print(f"     - 평균: {np.mean(type_diversities):.1f}개 타입")
print(f"     - 최대: {max(type_diversities)}개 타입")

# 5. 샘플 패턴 출력
print("\n5. 샘플 Community synthesis 패턴:")
for i, pattern in enumerate(community_synthesis_patterns[:3]):
    print(f"\n   패턴 {i+1}:")
    print(f"     커뮤니티: {pattern['community_title']}")
    print(f"     레벨: {pattern['level']}")
    print(f"     규모: {pattern['entity_count']}개 엔티티")
    print(f"     타입: {', '.join(pattern['entity_types'][:3])}...")
    print(f"     주요 엔티티: {', '.join(pattern['entities'][:3])}...")
    print(f"     요약: {pattern['community_summary'][:100]}...")
    if pattern['internal_relationships']:
        print(f"     내부 관계 예시: {pattern['internal_relationships'][0]['source']} → {pattern['internal_relationships'][0]['target']}")

print(f"\n✓ Community synthesis 패턴 추출 완료!")

### Cell 23: Community Synthesis 패턴 상세 분석

추출된 커뮤니티 분석 패턴을 상세히 분석합니다.

In [ ]:
print("=== Community Synthesis 패턴 상세 분석 ===\n")

print("1. 커뮤니티 구조 분석:")
print(f"   총 {len(community_synthesis_patterns)}개의 커뮤니티 패턴\n")

# 1. 레벨별 커뮤니티 특성 분석
level_characteristics = defaultdict(lambda: {
    'sizes': [],
    'type_counts': [],
    'density': [],
    'titles': []
})

for pattern in community_synthesis_patterns:
    level = pattern['level']
    level_characteristics[level]['sizes'].append(pattern['entity_count'])
    level_characteristics[level]['type_counts'].append(len(pattern['entity_types']))
    level_characteristics[level]['titles'].append(pattern['community_title'])
    
    # 커뮤니티 내부 연결 밀도 계산 (있는 관계 수 / 가능한 최대 관계 수)
    n_entities = len(pattern['entities'])
    n_internal_edges = len(pattern['internal_relationships'])
    max_possible_edges = n_entities * (n_entities - 1) / 2
    density = n_internal_edges / max_possible_edges if max_possible_edges > 0 else 0
    level_characteristics[level]['density'].append(density)

level_analysis = {}
print("2. 레벨별 커뮤니티 특성:")
for level in sorted(level_characteristics.keys()):
    chars = level_characteristics[level]
    print(f"\n   Level {level}:")
    print(f"     - 커뮤니티 수: {len(chars['sizes'])}개")
    print(f"     - 평균 크기: {np.mean(chars['sizes']):.1f} 엔티티")
    print(f"     - 평균 타입 다양성: {np.mean(chars['type_counts']):.1f}개")
    print(f"     - 평균 내부 밀도: {np.mean(chars['density']):.3f}")
    
    # 대표 커뮤니티 타이틀 샘플
    sample_titles = list(set(chars['titles']))[:3]
    print(f"     - 대표 커뮤니티: {', '.join(sample_titles)}")
    
    level_analysis[str(level)] = {  # JSON 직렬화를 위해 key를 string으로
        'community_count': len(chars['sizes']),
        'avg_size': float(np.mean(chars['sizes'])),
        'avg_type_diversity': float(np.mean(chars['type_counts'])),
        'avg_density': float(np.mean(chars['density'])),
        'sample_titles': sample_titles
    }

# 2. 커뮤니티 주제 분석
print("\n\n3. 커뮤니티 주제 패턴 분석:")

# 커뮤니티 타이틀에서 주요 키워드 추출
title_keywords = defaultdict(int)
for pattern in community_synthesis_patterns:
    title_words = pattern['community_title'].split()
    for word in title_words:
        if len(word) > 2:  # 2글자 이상만
            title_keywords[word] += 1

theme_keywords = []
print("   주요 커뮤니티 주제 키워드:")
for word, count in sorted(title_keywords.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"     - {word}: {count}회")
    theme_keywords.append({'keyword': word, 'count': count})

# 3. 엔티티 타입 조합 분석
print("\n\n4. 커뮤니티 내 엔티티 타입 조합:")

type_combinations = defaultdict(int)
for pattern in community_synthesis_patterns:
    # 타입들을 정렬하여 일관된 키 생성
    types_tuple = tuple(sorted(pattern['entity_types']))
    type_combinations[types_tuple] += 1

type_combo_analysis = []
print("   자주 나타나는 타입 조합:")
for types, count in sorted(type_combinations.items(), key=lambda x: x[1], reverse=True)[:5]:
    types_str = ' + '.join(types[:3]) + ('...' if len(types) > 3 else '')
    print(f"     - {types_str}: {count}개 커뮤니티")
    type_combo_analysis.append({
        'types': list(types),
        'count': count,
        'display': types_str
    })

# 4. 내부 관계 패턴 분석
print("\n\n5. 커뮤니티 내부 관계 패턴:")

relationship_patterns = {
    '협력/연계': [],
    '계층/분류': [],
    '인과/영향': [],
    '속성/특징': []
}

# 관계 설명에서 패턴 추출
for pattern in community_synthesis_patterns:
    for rel in pattern['internal_relationships']:
        desc = rel['description']
        if any(k in desc for k in ['협력', '연계', '함께', '공동']):
            relationship_patterns['협력/연계'].append((pattern, rel))
        elif any(k in desc for k in ['종류', '분류', '속한', '포함']):
            relationship_patterns['계층/분류'].append((pattern, rel))
        elif any(k in desc for k in ['영향', '때문', '인해', '결과']):
            relationship_patterns['인과/영향'].append((pattern, rel))
        elif any(k in desc for k in ['특징', '속성', '성질']):
            relationship_patterns['속성/특징'].append((pattern, rel))

rel_pattern_analysis = {}
print("   관계 유형별 분포:")
for rel_type, patterns in relationship_patterns.items():
    if patterns:
        print(f"\n   {rel_type}: {len(patterns)}개")
        # 샘플 출력
        sample_pattern, sample_rel = patterns[0]
        print(f"     커뮤니티: {sample_pattern['community_title']}")
        print(f"     관계: {sample_rel['source']} → {sample_rel['target']}")
        print(f"     설명: {sample_rel['description'][:50]}...")
        
        rel_pattern_analysis[rel_type] = {
            'count': len(patterns),
            'sample': {
                'community': sample_pattern['community_title'],
                'relation': f"{sample_rel['source']} → {sample_rel['target']}",
                'description': sample_rel['description']
            }
        }

# 5. 질문 생성 적합도 평가
print("\n\n6. Community Synthesis 질문 생성 적합도:")

high_quality_communities = []
quality_scores_comm = []
for pattern in community_synthesis_patterns:
    score = 0
    
    # 커뮤니티 크기 적절성 (너무 크거나 작지 않음)
    if 10 <= pattern['entity_count'] <= 30:
        score += 2
    elif 5 <= pattern['entity_count'] <= 40:
        score += 1
    
    # 타입 다양성
    if len(pattern['entity_types']) >= 3:
        score += 2
    elif len(pattern['entity_types']) >= 2:
        score += 1
    
    # 내부 관계 충분성
    if len(pattern['internal_relationships']) >= 4:
        score += 2
    elif len(pattern['internal_relationships']) >= 2:
        score += 1
    
    # 요약 품질 (길이 기준)
    if len(pattern['community_summary']) > 100:
        score += 1
    
    # 텍스트 증거
    if len(pattern['source_texts']) >= 3:
        score += 1
    
    pattern['synthesis_score'] = score
    quality_scores_comm.append(score)
    if score >= 5:
        high_quality_communities.append(pattern)

print(f"   고품질 커뮤니티: {len(high_quality_communities)}개 / {len(community_synthesis_patterns)}개")
print(f"   평균 품질 점수: {sum(p.get('synthesis_score', 0) for p in community_synthesis_patterns) / len(community_synthesis_patterns):.2f}")

# 6. 최고 품질 커뮤니티 예시
print("\n\n7. 최고 품질 Community Synthesis 패턴 예시:")
top_communities = sorted(community_synthesis_patterns, 
                        key=lambda x: x.get('synthesis_score', 0), 
                        reverse=True)[:3]

top_quality_communities = []
for i, pattern in enumerate(top_communities, 1):
    print(f"\n   예시 {i} (점수: {pattern.get('synthesis_score', 0)}):")
    print(f"   커뮤니티: {pattern['community_title']}")
    print(f"   레벨: {pattern['level']}")
    print(f"   규모: {pattern['entity_count']}개 엔티티 ({len(pattern['entity_types'])}개 타입)")
    print(f"   주요 엔티티:")
    
    key_entities = []
    for j, entity in enumerate(pattern['entities'][:5], 1):
        entity_type = G.nodes[entity]['type'] if entity in G else 'Unknown'
        print(f"     {j}. {entity} ({entity_type})")
        key_entities.append({'name': entity, 'type': entity_type})
    
    print(f"   요약: {pattern['community_summary'][:150]}...")
    
    print(f"   내부 관계 샘플:")
    internal_relations = []
    for j, rel in enumerate(pattern['internal_relationships'][:3], 1):
        print(f"     {j}. {rel['source']} → {rel['target']}")
        print(f"        {rel['description'][:60]}...")
        internal_relations.append({
            'source': rel['source'],
            'target': rel['target'],
            'description': rel['description']
        })
    
    top_quality_communities.append({
        'community_title': pattern['community_title'],
        'level': pattern['level'],
        'entity_count': pattern['entity_count'],
        'type_count': len(pattern['entity_types']),
        'synthesis_score': pattern.get('synthesis_score', 0),
        'key_entities': key_entities,
        'community_summary': pattern['community_summary'],
        'internal_relations': internal_relations
    })

# 분석 결과 저장
analysis_results_comm = {
    'pattern_count': len(community_synthesis_patterns),
    'level_analysis': level_analysis,
    'theme_keywords': theme_keywords[:10],
    'type_combinations': type_combo_analysis,
    'relationship_patterns': rel_pattern_analysis,
    'quality_distribution': {
        'high_quality_count': len(high_quality_communities),
        'average_score': float(np.mean(quality_scores_comm)),
        'score_distribution': pd.Series(quality_scores_comm).value_counts().to_dict()
    },
    'top_quality_examples': top_quality_communities
}

# JSON으로 저장
output_dir_comm = output_path / 'community_synthesis'
output_dir_comm.mkdir(exist_ok=True)

with open(output_dir_comm / 'community_pattern_analysis.json', 'w', encoding='utf-8') as f:
    json.dump(analysis_results_comm, f, ensure_ascii=False, indent=2)

# 패턴 자체도 저장 (추후 질문 생성에 사용)
with open(output_dir_comm / 'community_patterns.json', 'w', encoding='utf-8') as f:
    # JSON 직렬화를 위해 필요한 변환
    patterns_to_save = []
    for p in community_synthesis_patterns:
        pattern_copy = {
            'question_type': p['question_type'],
            'level': p['level'],
            'community_id': p['community_id'],
            'community_title': p['community_title'],
            'community_summary': p['community_summary'],
            'entities': p['entities'][:10],  # 상위 10개만
            'entity_count': p['entity_count'],
            'entity_types': p['entity_types'],
            'internal_relationships': p['internal_relationships'][:10],  # 상위 10개만
            'source_texts': list(p['source_texts'])[:5],  # 상위 5개만
            'pattern_template': p['pattern_template'],
            'difficulty': p['difficulty'],
            'synthesis_score': p.get('synthesis_score', 0)
        }
        patterns_to_save.append(pattern_copy)
    
    json.dump(patterns_to_save, f, ensure_ascii=False, indent=2)

print(f"\n✓ Community Synthesis 패턴 분석 결과 저장 완료:")
print(f"   - {output_dir_comm / 'community_pattern_analysis.json'}")
print(f"   - {output_dir_comm / 'community_patterns.json'}")
print("\n✓ Community Synthesis 패턴 상세 분석 완료!")

## Cell 24: Cross-context Integration 패턴 추출

여러 text unit에 걸쳐 통합해야 알 수 있는 패턴들을 추출합니다.

In [ ]:
print("=== Cross-context Integration 패턴 추출 ===")

# Cross-context integration 패턴 저장
cross_context_patterns = []

# 1. text unit별 엔티티/관계 분포 분석
print("\n1. Text unit별 지식 분포 분석:")
tu_stats = {}
sample_count = 0
for tu_id, entities in nodes_by_text_unit.items():
    if len(entities) > 2:  # 최소 2개 엔티티를 가진 text unit
        # 엔티티 타입 분석
        entity_types = defaultdict(int)
        for entity in entities:
            if entity in G:
                entity_types[G.nodes[entity]['type']] += 1
        
        tu_stats[tu_id] = {
            'entity_count': len(entities),
            'entity_types': dict(entity_types),
            'edge_count': len(edges_by_text_unit.get(tu_id, []))
        }
        
        # 샘플 출력 (처음 5개만)
        if sample_count < 5:
            print(f"   - TU {tu_id[:8]}...: {tu_stats[tu_id]['entity_count']} 엔티티, {tu_stats[tu_id]['edge_count']} 관계")
            sample_count += 1

print(f"\n   분석 가능한 text units: {len(tu_stats)}개 (전체 {len(nodes_by_text_unit)}개 중)")

# 2. text unit 간 공통 주제 찾기
print("\n2. Text unit 간 공통 주제 식별...")

# 엔티티 타입별로 여러 text unit에 걸쳐 있는 주제 찾기
cross_context_themes = defaultdict(lambda: defaultdict(set))

for entity_name in G.nodes():
    entity_data = G.nodes[entity_name]
    entity_type = entity_data['type']
    
    # 이 엔티티가 언급된 text units
    mentioned_units = set()
    for tu_id in entity_data['text_unit_ids']:
        mentioned_units.add(tu_id)
    
    # 2개 이상 text unit에 걸쳐 있는 경우
    if len(mentioned_units) >= 2:
        cross_context_themes[entity_type][entity_name] = mentioned_units

# 주요 cross-context 엔티티 타입
print("\n   Cross-context 엔티티 타입 분포:")
for entity_type, entities in sorted(cross_context_themes.items(), 
                                   key=lambda x: len(x[1]), reverse=True)[:10]:
    print(f"   - {entity_type}: {len(entities)}개 엔티티가 여러 text unit에 걸쳐 존재")

# 3. Cross-context 패턴 생성
print("\n3. Cross-context integration 패턴 생성 중...")

# 전략: 서로 다른 text unit에서 온 엔티티들이 연결된 경우 찾기
pattern_count = 0
for entity_type, cross_context_entities in cross_context_themes.items():
    if len(cross_context_entities) < 2:  # 너무 적은 경우 스킵
        continue
    
    # 이 타입의 주요 엔티티들 선택 (degree 기준)
    entities_with_degree = []
    for entity, units in list(cross_context_entities.items())[:50]:  # 상위 50개만
        if entity in G:
            entities_with_degree.append({
                'entity': entity,
                'degree': G.degree(entity),
                'units': list(units)
            })
    
    # degree 순으로 정렬
    entities_with_degree.sort(key=lambda x: x['degree'], reverse=True)
    
    # 상위 엔티티들로 패턴 생성
    for i, entity_info in enumerate(entities_with_degree[:20]):
        anchor_entity = entity_info['entity']
        anchor_units = entity_info['units']
        
        # 이 엔티티와 연결되어 있으면서 다른 text unit에서 온 엔티티 찾기
        related_entities = []
        for neighbor in G.neighbors(anchor_entity):
            neighbor_units = set(G.nodes[neighbor]['text_unit_ids'])
            
            # 다른 text unit에서 온 경우 (교집합이 없는 경우)
            if neighbor_units and not (neighbor_units & set(anchor_units)):
                edge_data = G.get_edge_data(anchor_entity, neighbor)
                if isinstance(edge_data, dict) and 0 in edge_data:
                    edge_data = edge_data[0]
                    
                related_entities.append({
                    'entity': neighbor,
                    'type': G.nodes[neighbor]['type'],
                    'units': list(neighbor_units)[:5],  # 최대 5개만
                    'relation': edge_data.get('description', ''),
                    'weight': edge_data.get('weight', 0)
                })
        
        if len(related_entities) >= 2:  # 충분한 cross-context 연결이 있는 경우
            # 텍스트 증거 수집
            source_texts = []
            units_involved = set(anchor_units[:5])  # anchor의 처음 5개 unit
            
            # anchor 엔티티 텍스트
            for text_info in G.nodes[anchor_entity].get('original_texts', [])[:2]:
                source_texts.append(text_info['text'])
            
            # 관련 엔티티들의 텍스트
            for rel_entity in related_entities[:3]:
                units_involved.update(rel_entity['units'][:2])
                entity_name = rel_entity['entity']
                if entity_name in G:
                    for text_info in G.nodes[entity_name].get('original_texts', [])[:1]:
                        source_texts.append(text_info['text'])
            
            pattern = {
                'question_type': 'cross_context',
                'anchor_entity': anchor_entity,
                'anchor_type': entity_type,
                'anchor_units': anchor_units[:5],  # 최대 5개
                'related_entities': related_entities[:5],  # 최대 5개
                'units_involved': list(units_involved),
                'unit_count': len(units_involved),
                'source_texts': source_texts[:5],  # 최대 5개
                'pattern_template': f"{anchor_entity}와 관련된 정보를 여러 컨텍스트에서 종합하면?",
                'difficulty': 'medium' if len(units_involved) <= 5 else 'hard'
            }
            
            cross_context_patterns.append(pattern)
            pattern_count += 1
            
            # 목표 패턴 수에 도달하면 중단
            if pattern_count >= 100:
                break
    
    if pattern_count >= 100:
        break

print(f"\n✓ {len(cross_context_patterns)}개 cross-context 패턴 추출")

# 4. 패턴 통계
print("\n4. Cross-context 패턴 통계:")

# text unit 수별 분포
unit_count_dist = defaultdict(int)
for pattern in cross_context_patterns:
    unit_count_dist[pattern['unit_count']] += 1

print("   Text unit 수별 분포:")
for unit_count in sorted(unit_count_dist.keys())[:10]:
    count = unit_count_dist[unit_count]
    print(f"     - {unit_count}개 units: {count}개 패턴 ({count/len(cross_context_patterns)*100:.1f}%)")

# anchor 엔티티 타입 분포
anchor_types = defaultdict(int)
for pattern in cross_context_patterns:
    anchor_types[pattern['anchor_type']] += 1

print("\n   Anchor 엔티티 타입 분포:")
for atype, count in sorted(anchor_types.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"     - {atype}: {count}개")

# 5. 샘플 패턴 출력
print("\n5. 샘플 Cross-context 패턴:")
for i, pattern in enumerate(cross_context_patterns[:3]):
    print(f"\n   패턴 {i+1}:")
    print(f"     Anchor: {pattern['anchor_entity']} ({pattern['anchor_type']})")
    print(f"     Anchor units: {len(pattern['anchor_units'])}개")
    print(f"     관련 엔티티 ({len(pattern['related_entities'])}개):")
    for j, rel in enumerate(pattern['related_entities'][:3]):
        print(f"       {j+1}. {rel['entity']} ({rel['type']})")
        print(f"          Units: {len(rel['units'])}개")
        print(f"          관계: {rel['relation'][:50]}...")
    print(f"     총 {pattern['unit_count']}개 text units 관련")

print(f"\n✓ Cross-context integration 패턴 추출 완료!")

### Cell 25: Cross-context Integration 패턴 상세 분석

추출된 Cross-context 패턴들의 text unit 간 연결성을 상세히 분석합니다.

In [ ]:
print("=== Cross-context Integration 패턴 상세 분석 ===\n")

print("1. 기본 통계:")
print(f"   총 {len(cross_context_patterns)}개의 cross-context 패턴\n")

# 1. text unit 관련성 분석
unit_involvement = defaultdict(int)
unit_pairs = defaultdict(int)
unit_clusters = []

for pattern in cross_context_patterns:
    units = pattern['units_involved']
    unit_clusters.append(set(units))
    
    # 각 text unit의 참여 횟수
    for unit in units:
        unit_involvement[unit] += 1
    
    # text unit 쌍 빈도
    for i, unit1 in enumerate(units):
        for unit2 in units[i+1:]:
            pair = tuple(sorted([unit1, unit2]))
            unit_pairs[pair] += 1

print("2. 가장 많이 참여하는 text units Top 10:")
for unit, count in sorted(unit_involvement.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"   - {unit[:12]}...: {count}개 패턴에 참여")

print("\n3. 자주 함께 나타나는 text unit 쌍 Top 10:")
for (unit1, unit2), count in sorted(unit_pairs.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"   - ({unit1[:8]}..., {unit2[:8]}...): {count}회")

# 2. 앵커 엔티티 분석
anchor_analysis = defaultdict(lambda: {
    'count': 0,
    'avg_related': 0,
    'avg_units': 0,
    'types': defaultdict(int)
})

for pattern in cross_context_patterns:
    anchor_type = pattern['anchor_type']
    anchor_analysis[anchor_type]['count'] += 1
    anchor_analysis[anchor_type]['avg_related'] += len(pattern['related_entities'])
    anchor_analysis[anchor_type]['avg_units'] += pattern['unit_count']
    
    # 관련 엔티티들의 타입
    for rel in pattern['related_entities']:
        anchor_analysis[anchor_type]['types'][rel['type']] += 1

print("\n4. 앵커 엔티티 타입별 분석:")
for anchor_type, stats in sorted(anchor_analysis.items(), 
                                key=lambda x: x[1]['count'], reverse=True)[:5]:
    count = stats['count']
    avg_related = stats['avg_related'] / count
    avg_units = stats['avg_units'] / count
    
    print(f"\n   {anchor_type} (총 {count}개 패턴):")
    print(f"     - 평균 관련 엔티티: {avg_related:.1f}개")
    print(f"     - 평균 text unit 수: {avg_units:.1f}개")
    print(f"     - 주요 관련 타입:")
    for rtype, rcount in sorted(stats['types'].items(), 
                                key=lambda x: x[1], reverse=True)[:3]:
        print(f"       · {rtype}: {rcount}회")

# 3. text unit 간 지식 통합 패턴 분석
integration_patterns = {
    '근거리 통합': [],    # 같은 문서 내 가까운 unit들
    '원거리 통합': [],    # 같은 문서 내 먼 unit들
    '교차 문서 통합': [], # 다른 문서의 unit들
    '다중 컨텍스트': []  # 3개 이상의 다양한 unit들
}

# 패턴 분류
for pattern in cross_context_patterns:
    units = pattern['units_involved']
    unit_count = len(units)
    
    # 문서 정보 추출 (text_unit_to_doc 사용)
    docs_involved = set()
    for unit in units:
        if unit in text_unit_to_doc:
            docs_involved.add(text_unit_to_doc[unit])
    
    # 분류 로직
    if unit_count >= 5:
        integration_patterns['다중 컨텍스트'].append(pattern)
    elif len(docs_involved) > 1:
        integration_patterns['교차 문서 통합'].append(pattern)
    elif unit_count <= 3:
        integration_patterns['근거리 통합'].append(pattern)
    else:
        integration_patterns['원거리 통합'].append(pattern)

print("\n5. Text unit 간 지식 통합 패턴:")
for pattern_type, patterns in integration_patterns.items():
    if patterns:
        print(f"\n   {pattern_type}: {len(patterns)}개")
        if patterns:
            sample = patterns[0]
            print(f"     예시: {sample['anchor_entity']} ({sample['anchor_type']})")
            print(f"     Units: {sample['unit_count']}개")

# 4. 질문 생성 적합도 평가
high_quality_cross_context = []
quality_scores_cross = []

for pattern in cross_context_patterns:
    score = 0
    
    # text unit 다양성
    if pattern['unit_count'] >= 5:
        score += 2
    elif pattern['unit_count'] >= 3:
        score += 1
    
    # 관련 엔티티 충분성
    if len(pattern['related_entities']) >= 4:
        score += 2
    elif len(pattern['related_entities']) >= 2:
        score += 1
    
    # 엔티티 타입 다양성
    unique_types = len(set([pattern['anchor_type']] + 
                          [r['type'] for r in pattern['related_entities']]))
    if unique_types >= 3:
        score += 2
    elif unique_types >= 2:
        score += 1
    
    # 텍스트 증거
    if len(pattern['source_texts']) >= 4:
        score += 1
    
    pattern['cross_context_score'] = score
    quality_scores_cross.append(score)
    if score >= 5:
        high_quality_cross_context.append(pattern)

print(f"\n6. Cross-context 질문 생성 적합도:")
print(f"   고품질 패턴: {len(high_quality_cross_context)}개 / {len(cross_context_patterns)}개")
print(f"   평균 품질 점수: {np.mean(quality_scores_cross):.2f}")

# 5. 최고 품질 패턴 예시
print("\n7. 최고 품질 Cross-context 패턴 예시:")
top_cross_context = sorted(cross_context_patterns, 
                          key=lambda x: x.get('cross_context_score', 0), 
                          reverse=True)[:3]

top_quality_cross_context = []
for i, pattern in enumerate(top_cross_context, 1):
    print(f"\n   예시 {i} (점수: {pattern.get('cross_context_score', 0)}):")
    print(f"   앵커: {pattern['anchor_entity']} ({pattern['anchor_type']})")
    print(f"   앵커 text units: {len(pattern['anchor_units'])}개")
    print(f"\n   관련 엔티티 네트워크:")
    
    related_info = []
    for j, rel in enumerate(pattern['related_entities'][:5], 1):
        print(f"     {j}. {rel['entity']} ({rel['type']})")
        print(f"        - Text units: {len(rel['units'])}개")
        print(f"        - 관계: {rel['relation'][:60]}...")
        print(f"        - 가중치: {rel['weight']}")
        
        related_info.append({
            'entity': rel['entity'],
            'type': rel['type'],
            'units': rel['units'],
            'relation': rel['relation'],
            'weight': rel['weight']
        })
    
    print(f"\n   통합 정보:")
    print(f"     - 총 {pattern['unit_count']}개 text units 통합 필요")
    print(f"     - 텍스트 증거: {len(pattern['source_texts'])}개")
    
    top_quality_cross_context.append({
        'anchor_entity': pattern['anchor_entity'],
        'anchor_type': pattern['anchor_type'],
        'anchor_units': pattern['anchor_units'],
        'related_entities': related_info,
        'unit_count': pattern['unit_count'],
        'units_involved': pattern['units_involved'],
        'cross_context_score': pattern.get('cross_context_score', 0)
    })

# 분석 결과 저장
analysis_results_cross = {
    'pattern_count': len(cross_context_patterns),
    'unit_statistics': {
        'most_involved_units': sorted(unit_involvement.items(), 
                                     key=lambda x: x[1], reverse=True)[:10],
        'common_unit_pairs': sorted(unit_pairs.items(), 
                                   key=lambda x: x[1], reverse=True)[:10]
    },
    'anchor_analysis': {
        anchor_type: {
            'count': stats['count'],
            'avg_related_entities': stats['avg_related'] / stats['count'],
            'avg_unit_count': stats['avg_units'] / stats['count'],
            'top_related_types': sorted(stats['types'].items(), 
                                      key=lambda x: x[1], reverse=True)[:5]
        }
        for anchor_type, stats in sorted(anchor_analysis.items(), 
                                       key=lambda x: x[1]['count'], reverse=True)[:5]
    },
    'integration_patterns': {
        ptype: len(patterns) for ptype, patterns in integration_patterns.items()
    },
    'quality_distribution': {
        'high_quality_count': len(high_quality_cross_context),
        'average_score': float(np.mean(quality_scores_cross)),
        'score_distribution': pd.Series(quality_scores_cross).value_counts().to_dict()
    },
    'top_quality_examples': top_quality_cross_context
}

# JSON으로 저장 - 경로 결정
output_dir_cross = output_path / 'cross_context'  # 새로운 폴더명 사용
output_dir_cross.mkdir(exist_ok=True)

with open(output_dir_cross / 'cross_context_pattern_analysis.json', 'w', encoding='utf-8') as f:
    json.dump(analysis_results_cross, f, ensure_ascii=False, indent=2)

# 패턴 자체도 저장
with open(output_dir_cross / 'cross_context_patterns.json', 'w', encoding='utf-8') as f:
    patterns_to_save = []
    for p in cross_context_patterns:
        pattern_copy = {
            'question_type': p['question_type'],
            'anchor_entity': p['anchor_entity'],
            'anchor_type': p['anchor_type'],
            'anchor_units': p['anchor_units'],
            'related_entities': p['related_entities'][:10],  # 최대 10개
            'units_involved': p['units_involved'],
            'unit_count': p['unit_count'],
            'source_texts': p['source_texts'][:5],  # 최대 5개
            'pattern_template': p['pattern_template'],
            'difficulty': p['difficulty'],
            'cross_context_score': p.get('cross_context_score', 0)
        }
        patterns_to_save.append(pattern_copy)
    
    json.dump(patterns_to_save, f, ensure_ascii=False, indent=2)

print(f"\n✓ Cross-context 패턴 분석 결과 저장 완료:")
print(f"   - {output_dir_cross / 'cross_context_pattern_analysis.json'}")
print(f"   - {output_dir_cross / 'cross_context_patterns.json'}")
print("\n✓ Cross-context Integration 패턴 상세 분석 완료!")

## Cell 25: Implicit Relationships 패턴 추출

그래프에서 직접 연결되지 않았지만 커뮤니티나 공통 이웃을 통해 암시적으로 연결된 관계를 찾습니다.

In [ ]:
print("=== Implicit Relationships 패턴 추출 ===")

# Implicit relationships 패턴 저장
implicit_patterns = []

# 1. 커뮤니티 기반 암시적 관계 추출
print("\n1. 커뮤니티 기반 암시적 관계 찾기...")

community_based_patterns = []

# 각 커뮤니티에서 직접 연결되지 않은 노드 쌍 찾기
for level in sorted(nodes_by_community.keys())[:3]:  # 상위 3개 레벨만
    for comm_id, nodes in nodes_by_community[level].items():
        if len(nodes) < 5 or len(nodes) > 50:  # 적절한 크기의 커뮤니티만
            continue
            
        # 커뮤니티 리포트 가져오기
        report_key = (comm_id, level)
        report = community_reports_lookup.get(report_key, {})
        
        if not report.get('summary'):  # 요약이 없으면 스킵
            continue
            
        # 중요도 높은 노드 선택 (PageRank 기준)
        nodes_with_pr = [(n, centrality_measures['pagerank'].get(n, 0)) 
                         for n in nodes if n in G]
        nodes_with_pr.sort(key=lambda x: x[1], reverse=True)
        top_nodes = [n for n, _ in nodes_with_pr[:10]]  # 상위 10개 노드
        
        # 직접 연결되지 않은 노드 쌍 찾기
        for i, node1 in enumerate(top_nodes):
            for node2 in top_nodes[i+1:]:
                # 직접 연결 체크
                if not G.has_edge(node1, node2) and not G.has_edge(node2, node1):
                    # 공통 이웃 확인
                    common = set(G.neighbors(node1)) & set(G.neighbors(node2))
                    
                    # 텍스트 증거 수집
                    texts = []
                    for node in [node1, node2]:
                        for text_info in G.nodes[node].get('original_texts', [])[:1]:
                            texts.append(text_info['text'])
                    
                    # 커뮤니티 요약도 텍스트 증거로 추가
                    texts.append(report['summary'][:500])
                    
                    pattern = {
                        'question_type': 'implicit_relationships',
                        'pattern_subtype': 'community_based',
                        'entities': [node1, node2],
                        'entity_types': [G.nodes[node1]['type'], G.nodes[node2]['type']],
                        'community_id': comm_id,
                        'community_level': level,
                        'community_title': report.get('title', ''),
                        'community_summary': report.get('summary', '')[:200],
                        'common_neighbors': list(common)[:5],  # 최대 5개
                        'common_neighbor_count': len(common),
                        'source_texts': texts[:4],  # 최대 4개
                        'pattern_template': f"{node1}과 {node2}의 {report.get('title', '커뮤니티')} 내에서의 연관성은?",
                        'difficulty': 'medium'
                    }
                    
                    community_based_patterns.append(pattern)
                    
                    if len(community_based_patterns) >= 60:  # 목표 개수
                        break
                        
            if len(community_based_patterns) >= 60:
                break
                
        if len(community_based_patterns) >= 60:
            break
            
    if len(community_based_patterns) >= 60:
        break

print(f"✓ 커뮤니티 기반 암시적 관계 {len(community_based_patterns)}개 추출")

# 2. 공통 이웃 기반 암시적 관계 추출
print("\n2. 공통 이웃 기반 암시적 관계 찾기...")

common_neighbor_patterns = []

# PageRank 상위 노드들 중심으로 탐색
top_nodes_pr = sorted(centrality_measures['pagerank'].items(), 
                     key=lambda x: x[1], reverse=True)[:100]

for node1, pr1 in top_nodes_pr[:50]:
    if node1 not in G:
        continue
        
    # node1의 이웃들
    neighbors1 = set(G.neighbors(node1))
    
    for node2, pr2 in top_nodes_pr[50:100]:  # 다른 그룹에서 선택
        if node2 not in G or node2 == node1:
            continue
            
        # 직접 연결되어 있으면 스킵
        if G.has_edge(node1, node2) or G.has_edge(node2, node1):
            continue
            
        # 공통 이웃 확인
        common = neighbors1 & set(G.neighbors(node2))
        
        if len(common) >= 2:  # 최소 2개 이상의 공통 이웃
            # 공통 이웃들의 타입 분석
            common_types = defaultdict(int)
            common_relations = []
            
            for neighbor in list(common)[:5]:  # 상위 5개만
                if neighbor in G:
                    common_types[G.nodes[neighbor]['type']] += 1
                    
                    # 관계 설명 수집
                    edge1 = G.get_edge_data(node1, neighbor)
                    edge2 = G.get_edge_data(node2, neighbor)
                    
                    if isinstance(edge1, dict) and 0 in edge1:
                        edge1 = edge1[0]
                    if isinstance(edge2, dict) and 0 in edge2:
                        edge2 = edge2[0]
                        
                    if edge1 and edge2:
                        common_relations.append({
                            'neighbor': neighbor,
                            'relation_to_node1': edge1.get('description', '')[:50],
                            'relation_to_node2': edge2.get('description', '')[:50]
                        })
            
            # 텍스트 증거 수집
            texts = []
            for node in [node1, node2] + list(common)[:2]:
                if node in G:
                    for text_info in G.nodes[node].get('original_texts', [])[:1]:
                        texts.append(text_info['text'])
            
            pattern = {
                'question_type': 'implicit_relationships',
                'pattern_subtype': 'common_neighbor_based',
                'entities': [node1, node2],
                'entity_types': [G.nodes[node1]['type'], G.nodes[node2]['type']],
                'common_neighbors': list(common)[:10],  # 최대 10개
                'common_neighbor_count': len(common),
                'common_neighbor_types': dict(common_types),
                'common_relations': common_relations[:3],  # 최대 3개
                'source_texts': texts[:4],  # 최대 4개
                'pattern_template': f"{node1}과 {node2}가 공통으로 관련된 {len(common)}개 요소를 통한 간접적 관계는?",
                'difficulty': 'hard' if len(common) >= 5 else 'medium'
            }
            
            common_neighbor_patterns.append(pattern)
            
            if len(common_neighbor_patterns) >= 30:  # 목표 개수
                break
                
    if len(common_neighbor_patterns) >= 30:
        break

print(f"✓ 공통 이웃 기반 암시적 관계 {len(common_neighbor_patterns)}개 추출")

# 3. 복합적 암시 관계 추출 (커뮤니티 + 공통 이웃)
print("\n3. 복합적 암시 관계 찾기...")

complex_implicit_patterns = []

# 이미 찾은 패턴들에서 더 복잡한 관계 찾기
for comm_pattern in community_based_patterns[:20]:
    node1, node2 = comm_pattern['entities']
    
    # 이 노드들이 다른 커뮤니티에서도 연결되는지 확인
    shared_communities = []
    for level in nodes_by_community.keys():
        for comm_id, nodes in nodes_by_community[level].items():
            if node1 in nodes and node2 in nodes:
                shared_communities.append((level, comm_id))
    
    if len(shared_communities) >= 2:  # 여러 커뮤니티에 공존
        # 추가 맥락 수집
        multi_community_context = []
        for level, comm_id in shared_communities[:3]:
            report = community_reports_lookup.get((comm_id, level), {})
            if report:
                multi_community_context.append({
                    'level': level,
                    'community_id': comm_id,
                    'title': report.get('title', '')[:50],
                    'summary': report.get('summary', '')[:100]
                })
        
        pattern = {
            'question_type': 'implicit_relationships',
            'pattern_subtype': 'complex_multi_context',
            'entities': [node1, node2],
            'entity_types': comm_pattern['entity_types'],
            'primary_community': {
                'id': comm_pattern['community_id'],
                'level': comm_pattern['community_level'],
                'title': comm_pattern['community_title']
            },
            'shared_communities': multi_community_context,
            'community_count': len(shared_communities),
            'common_neighbors': comm_pattern['common_neighbors'],
            'source_texts': comm_pattern['source_texts'],
            'pattern_template': f"{node1}과 {node2}가 {len(shared_communities)}개 커뮤니티에서 보이는 다면적 연관성은?",
            'difficulty': 'hard'
        }
        
        complex_implicit_patterns.append(pattern)
        
        if len(complex_implicit_patterns) >= 10:
            break

print(f"✓ 복합적 암시 관계 {len(complex_implicit_patterns)}개 추출")

# 4. 모든 암시적 관계 패턴 통합
implicit_patterns = community_based_patterns + common_neighbor_patterns + complex_implicit_patterns

print(f"\n4. 총 {len(implicit_patterns)}개 implicit relationships 패턴 추출")
print(f"   - 커뮤니티 기반: {len(community_based_patterns)}개")
print(f"   - 공통 이웃 기반: {len(common_neighbor_patterns)}개")
print(f"   - 복합적 암시 관계: {len(complex_implicit_patterns)}개")

# 5. 샘플 패턴 출력
print("\n5. 샘플 Implicit Relationships 패턴:")

# 각 유형별로 하나씩
for subtype, patterns in [
    ('community_based', community_based_patterns),
    ('common_neighbor_based', common_neighbor_patterns),
    ('complex_multi_context', complex_implicit_patterns)
]:
    if patterns:
        pattern = patterns[0]
        print(f"\n   [{subtype}]")
        print(f"   엔티티: {pattern['entities'][0]} ↔ {pattern['entities'][1]}")
        print(f"   타입: {pattern['entity_types'][0]} ↔ {pattern['entity_types'][1]}")
        
        if subtype == 'community_based':
            print(f"   커뮤니티: {pattern['community_title']}")
            print(f"   공통 이웃 수: {pattern['common_neighbor_count']}")
        elif subtype == 'common_neighbor_based':
            print(f"   공통 이웃: {pattern['common_neighbor_count']}개")
            if pattern['common_relations']:
                rel = pattern['common_relations'][0]
                print(f"   예시: {rel['neighbor']}를 통한 연결")
        else:
            print(f"   공유 커뮤니티: {pattern['community_count']}개")

print(f"\n✓ Implicit relationships 패턴 추출 완료!")

### Cell 27: Implicit Relationships 패턴 상세 분석

추출된 암시적 관계 패턴들의 구조와 특성을 상세히 분석합니다.

In [ ]:
print("=== Implicit Relationships 패턴 상세 분석 ===\n")

print("1. 기본 통계:")
print(f"   총 {len(implicit_patterns)}개의 implicit relationships 패턴")
print(f"   - 커뮤니티 기반: {len(community_based_patterns)}개")
print(f"   - 공통 이웃 기반: {len(common_neighbor_patterns)}개")
print(f"   - 복합적 암시 관계: {len(complex_implicit_patterns)}개\n")

# 1. 커뮤니티 기반 암시적 관계 분석
print("2. 커뮤니티 기반 암시적 관계 분석:")

# 커뮤니티 레벨별 분포
comm_level_dist = defaultdict(int)
comm_sizes = []
comm_titles_sample = []

for pattern in community_based_patterns:
    comm_level_dist[pattern['community_level']] += 1
    
    # 커뮤니티 정보 수집
    if pattern['community_title'] not in comm_titles_sample and len(comm_titles_sample) < 10:
        comm_titles_sample.append(pattern['community_title'])

print("\n   2.1. 커뮤니티 레벨별 분포:")
for level in sorted(comm_level_dist.keys()):
    count = comm_level_dist[level]
    print(f"       Level {level}: {count}개 패턴 ({count/len(community_based_patterns)*100:.1f}%)")

print("\n   2.2. 주요 커뮤니티 테마 (샘플):")
for i, title in enumerate(comm_titles_sample[:5], 1):
    print(f"       {i}. {title}")

# 엔티티 타입 조합 분석
entity_type_pairs = defaultdict(int)
for pattern in community_based_patterns:
    type_pair = tuple(sorted(pattern['entity_types']))
    entity_type_pairs[type_pair] += 1

print("\n   2.3. 주요 엔티티 타입 조합:")
for (type1, type2), count in sorted(entity_type_pairs.items(), 
                                   key=lambda x: x[1], reverse=True)[:5]:
    print(f"       {type1} ↔ {type2}: {count}회")

# 공통 이웃 수 분포
common_neighbor_counts = [p['common_neighbor_count'] for p in community_based_patterns]
print(f"\n   2.4. 공통 이웃 수 분포:")
print(f"       평균: {np.mean(common_neighbor_counts):.1f}개")
print(f"       중앙값: {np.median(common_neighbor_counts):.0f}개")
print(f"       최대: {max(common_neighbor_counts)}개")

# 2. 공통 이웃 기반 암시적 관계 분석
print("\n\n3. 공통 이웃 기반 암시적 관계 분석:")

if common_neighbor_patterns:
    # 공통 이웃 수에 따른 패턴 분류
    neighbor_count_dist = defaultdict(list)
    for pattern in common_neighbor_patterns:
        count = pattern['common_neighbor_count']
        if count <= 2:
            category = '소규모 (2개)'
        elif count <= 5:
            category = '중규모 (3-5개)'
        else:
            category = '대규모 (6개+)'
        neighbor_count_dist[category].append(pattern)
    
    print("\n   3.1. 공통 이웃 규모별 분포:")
    for category in ['소규모 (2개)', '중규모 (3-5개)', '대규모 (6개+)']:
        patterns = neighbor_count_dist.get(category, [])
        print(f"       {category}: {len(patterns)}개 패턴")
    
    # 공통 이웃들의 타입 분석
    all_neighbor_types = defaultdict(int)
    for pattern in common_neighbor_patterns:
        if 'common_neighbor_types' in pattern:
            for ntype, count in pattern['common_neighbor_types'].items():
                all_neighbor_types[ntype] += count
    
    print("\n   3.2. 공통 이웃들의 주요 타입:")
    for ntype, count in sorted(all_neighbor_types.items(), 
                              key=lambda x: x[1], reverse=True)[:5]:
        print(f"       {ntype}: {count}회")
    
    # 관계 패턴 분석
    relation_patterns_analysis = []
    for pattern in common_neighbor_patterns[:10]:  # 상위 10개만
        if pattern['common_relations']:
            rel = pattern['common_relations'][0]
            relation_patterns_analysis.append({
                'entities': pattern['entities'],
                'neighbor': rel['neighbor'],
                'rel1': rel['relation_to_node1'],
                'rel2': rel['relation_to_node2']
            })
    
    if relation_patterns_analysis:
        print("\n   3.3. 공통 이웃을 통한 관계 패턴 예시:")
        for i, rel_info in enumerate(relation_patterns_analysis[:3], 1):
            print(f"\n       예시 {i}:")
            print(f"       {rel_info['entities'][0]} ← {rel_info['neighbor']} → {rel_info['entities'][1]}")
            print(f"       관계1: {rel_info['rel1']}")
            print(f"       관계2: {rel_info['rel2']}")

# 3. 복합적 암시 관계 분석
print("\n\n4. 복합적 암시 관계 분석:")

if complex_implicit_patterns:
    # 공유 커뮤니티 수 분포
    shared_comm_counts = [p['community_count'] for p in complex_implicit_patterns]
    print(f"\n   4.1. 공유 커뮤니티 수:")
    print(f"       평균: {np.mean(shared_comm_counts):.1f}개")
    print(f"       최대: {max(shared_comm_counts)}개")
    
    # 다층적 관계 예시
    print("\n   4.2. 다층적 관계 예시:")
    for i, pattern in enumerate(complex_implicit_patterns[:2], 1):
        print(f"\n       예시 {i}:")
        print(f"       엔티티: {pattern['entities'][0]} ↔ {pattern['entities'][1]}")
        print(f"       주 커뮤니티: {pattern['primary_community']['title']}")
        print(f"       공유 커뮤니티 수: {pattern['community_count']}개")
        if pattern['shared_communities']:
            print(f"       다른 맥락:")
            for j, comm in enumerate(pattern['shared_communities'][:2], 1):
                print(f"         {j}. {comm['title']} (Level {comm['level']})")

# 4. 암시적 관계의 강도 평가
print("\n\n5. 암시적 관계 강도 평가:")

# 각 패턴에 암시적 관계 강도 점수 부여
implicit_strength_scores = []
high_strength_patterns = []

for pattern in implicit_patterns:
    score = 0
    
    # 공통 이웃 수
    common_count = pattern.get('common_neighbor_count', 0)
    if common_count >= 5:
        score += 3
    elif common_count >= 3:
        score += 2
    elif common_count >= 1:
        score += 1
    
    # 커뮤니티 컨텍스트
    if pattern['pattern_subtype'] == 'community_based':
        score += 2  # 커뮤니티 맥락 보너스
    elif pattern['pattern_subtype'] == 'complex_multi_context':
        score += 3 * pattern.get('community_count', 1)  # 다중 커뮤니티 보너스
    
    # 엔티티 타입 관련성
    if pattern['entity_types'][0] == pattern['entity_types'][1]:
        score += 1  # 같은 타입
    
    # 텍스트 증거
    if len(pattern['source_texts']) >= 3:
        score += 1
    
    pattern['implicit_strength'] = score
    implicit_strength_scores.append(score)
    
    if score >= 6:
        high_strength_patterns.append(pattern)

print(f"   고강도 암시적 관계: {len(high_strength_patterns)}개 / {len(implicit_patterns)}개")
print(f"   평균 강도 점수: {np.mean(implicit_strength_scores):.2f}")
print(f"   점수 분포:")
score_dist = pd.Series(implicit_strength_scores).value_counts().sort_index()
for score, count in score_dist.items():
    print(f"     {score}점: {count}개")

# 5. 최고 품질 암시적 관계 패턴
print("\n\n6. 최고 품질 암시적 관계 패턴 예시:")
top_implicit = sorted(implicit_patterns, 
                     key=lambda x: x.get('implicit_strength', 0), 
                     reverse=True)[:5]

top_quality_implicit = []
for i, pattern in enumerate(top_implicit, 1):
    print(f"\n   예시 {i} (강도: {pattern.get('implicit_strength', 0)}):")
    print(f"   유형: {pattern['pattern_subtype']}")
    print(f"   엔티티: {pattern['entities'][0]} ↔ {pattern['entities'][1]}")
    print(f"   타입: {pattern['entity_types'][0]} ↔ {pattern['entity_types'][1]}")
    
    if pattern['pattern_subtype'] == 'community_based':
        print(f"   커뮤니티: {pattern['community_title']}")
        print(f"   커뮤니티 레벨: {pattern['community_level']}")
    
    # common_neighbor_count 키 존재 여부 확인
    if 'common_neighbor_count' in pattern:
        print(f"   공통 이웃: {pattern['common_neighbor_count']}개")
        if pattern.get('common_neighbors'):
            print(f"   주요 공통 이웃: {', '.join(pattern['common_neighbors'][:3])}")
    
    if pattern['pattern_subtype'] == 'complex_multi_context':
        print(f"   공유 커뮤니티: {pattern.get('community_count', 0)}개")
    
    print(f"   템플릿: {pattern['pattern_template']}")
    
    top_quality_implicit.append({
        'pattern_subtype': pattern['pattern_subtype'],
        'entities': pattern['entities'],
        'entity_types': pattern['entity_types'],
        'implicit_strength': pattern.get('implicit_strength', 0),
        'common_neighbor_count': pattern.get('common_neighbor_count', 0),
        'community_info': {
            'title': pattern.get('community_title', ''),
            'level': pattern.get('community_level', -1)
        } if pattern['pattern_subtype'] == 'community_based' else None
    })

# 6. GraphRAG의 장점 분석
print("\n\n7. GraphRAG의 암시적 관계 탐지 장점:")
print("\n   전통적 RAG vs GraphRAG:")
print("   - 전통적 RAG: 텍스트 내 명시적 언급만 찾을 수 있음")
print("   - GraphRAG: 커뮤니티 구조와 네트워크 토폴로지를 통해 암시적 연결 발견")

# 암시적 관계 유형별 예시
print("\n   발견된 암시적 관계 유형:")
print(f"   1. 커뮤니티 맥락 기반: {len(community_based_patterns)}개")
print("      → 같은 주제나 도메인 내에서의 숨은 연관성")
print(f"   2. 네트워크 구조 기반: {len(common_neighbor_patterns)}개")
print("      → 공통 연결고리를 통한 간접적 관계")
print(f"   3. 다층적 복합 관계: {len(complex_implicit_patterns)}개")
print("      → 여러 관점에서 나타나는 복잡한 연관성")

# 분석 결과 저장
analysis_results_implicit = {
    'pattern_count': len(implicit_patterns),
    'pattern_distribution': {
        'community_based': len(community_based_patterns),
        'common_neighbor_based': len(common_neighbor_patterns),
        'complex_multi_context': len(complex_implicit_patterns)
    },
    'community_analysis': {
        'level_distribution': dict(comm_level_dist),
        'sample_titles': comm_titles_sample[:10],
        'entity_type_pairs': [(list(pair), count) for pair, count in 
                              sorted(entity_type_pairs.items(), 
                                    key=lambda x: x[1], reverse=True)[:10]]
    },
    'common_neighbor_analysis': {
        'average_count': float(np.mean([p.get('common_neighbor_count', 0) 
                                       for p in common_neighbor_patterns])) if common_neighbor_patterns else 0,
        'neighbor_types': dict(all_neighbor_types),
        'size_distribution': {k: len(v) for k, v in neighbor_count_dist.items()}
    },
    'strength_analysis': {
        'high_strength_count': len(high_strength_patterns),
        'average_strength': float(np.mean(implicit_strength_scores)),
        'score_distribution': score_dist.to_dict()
    },
    'top_quality_examples': top_quality_implicit
}

# JSON으로 저장
output_dir_implicit = output_path / 'implicit_relationships'
output_dir_implicit.mkdir(exist_ok=True)

with open(output_dir_implicit / 'implicit_pattern_analysis.json', 'w', encoding='utf-8') as f:
    json.dump(analysis_results_implicit, f, ensure_ascii=False, indent=2)

# 패턴 자체도 저장
with open(output_dir_implicit / 'implicit_patterns.json', 'w', encoding='utf-8') as f:
    patterns_to_save = []
    for p in implicit_patterns:
        pattern_copy = {
            'question_type': p['question_type'],
            'pattern_subtype': p['pattern_subtype'],
            'entities': p['entities'],
            'entity_types': p['entity_types'],
            'common_neighbors': p.get('common_neighbors', [])[:10],
            'common_neighbor_count': p.get('common_neighbor_count', 0),
            'source_texts': p['source_texts'][:5],
            'pattern_template': p['pattern_template'],
            'difficulty': p['difficulty'],
            'implicit_strength': p.get('implicit_strength', 0)
        }
        
        # 서브타입별 추가 정보
        if p['pattern_subtype'] == 'community_based':
            pattern_copy.update({
                'community_id': p['community_id'],
                'community_level': p['community_level'],
                'community_title': p['community_title'],
                'community_summary': p.get('community_summary', '')[:200]
            })
        elif p['pattern_subtype'] == 'common_neighbor_based':
            pattern_copy.update({
                'common_neighbor_types': p.get('common_neighbor_types', {}),
                'common_relations': p.get('common_relations', [])[:5]
            })
        elif p['pattern_subtype'] == 'complex_multi_context':
            pattern_copy.update({
                'primary_community': p.get('primary_community', {}),
                'shared_communities': p.get('shared_communities', [])[:5],
                'community_count': p.get('community_count', 0)
            })
        
        patterns_to_save.append(pattern_copy)
    
    json.dump(patterns_to_save, f, ensure_ascii=False, indent=2)

print(f"\n✓ Implicit relationships 패턴 분석 결과 저장 완료:")
print(f"   - {output_dir_implicit / 'implicit_pattern_analysis.json'}")
print(f"   - {output_dir_implicit / 'implicit_patterns.json'}")
print("\n✓ Implicit Relationships 패턴 상세 분석 완료!")

## Cell 28: Causal Chains 패턴 추출

원인과 결과의 연쇄적 관계를 형성하는 인과관계 체인을 추출합니다.

In [ ]:
print("=== Causal Chains 패턴 추출 ===")

# Causal chains 패턴 저장
causal_chains_patterns = []

# 인과관계 키워드 정의
causal_keywords = {
    'cause': ['때문', '인해', '인한', '원인', '유발', '초래', '야기'],
    'effect': ['결과', '따라', '영향', '효과', '변화', '증가', '감소', '개선', '악화'],
    'mechanism': ['통해', '으로써', '의해', '거쳐', '경유']
}

# 모든 키워드를 하나의 리스트로
all_causal_keywords = []
for keywords in causal_keywords.values():
    all_causal_keywords.extend(keywords)

# 1. 직접적 인과관계 찾기 (1-hop causal)
print("\n1. 직접적 인과관계 추출...")
direct_causal_patterns = []

# 모든 엣지 검사
for u, v, data in G.edges(data=True):
    description = data.get('description', '')
    
    # 인과관계 키워드 체크
    if any(keyword in description for keyword in all_causal_keywords):
        # 인과관계 타입 분류
        causal_type = 'general'
        for ctype, keywords in causal_keywords.items():
            if any(k in description for k in keywords):
                causal_type = ctype
                break
        
        # 텍스트 증거 수집
        texts = []
        for node in [u, v]:
            if node in G:
                for text_info in G.nodes[node].get('original_texts', [])[:1]:
                    texts.append(text_info['text'])
        
        # 엣지의 원본 텍스트도 추가
        if 'original_texts' in data:
            texts.extend(data['original_texts'][:1])
        
        pattern = {
            'question_type': 'causal_chains',
            'pattern_subtype': 'direct_causal',
            'chain_length': 1,
            'entities': [u, v],
            'entity_types': [G.nodes[u]['type'], G.nodes[v]['type']],
            'causal_relation': {
                'cause': u,
                'effect': v,
                'description': description,
                'weight': data.get('weight', 0),
                'type': causal_type
            },
            'source_texts': texts[:3],
            'pattern_template': f"{u}이(가) {v}에 미치는 인과적 영향은?",
            'difficulty': 'easy'
        }
        
        direct_causal_patterns.append(pattern)

# 가중치 높은 순으로 정렬하고 상위 N개만 선택
direct_causal_patterns.sort(key=lambda x: x['causal_relation']['weight'], reverse=True)
direct_causal_patterns = direct_causal_patterns[:50]  # 상위 50개

print(f"✓ 직접적 인과관계 {len(direct_causal_patterns)}개 추출")

# 2. 인과관계 체인 찾기 (2-3 hop causal chains)
print("\n2. 인과관계 체인 추출...")
causal_chain_patterns = []

# 직접적 인과관계를 시작점으로 체인 확장
for direct_pattern in direct_causal_patterns[:30]:  # 상위 30개에서 시작
    effect_node = direct_pattern['entities'][1]
    
    # effect_node에서 나가는 인과관계 찾기
    for next_node in G.neighbors(effect_node):
        edge_data = G.get_edge_data(effect_node, next_node)
        if isinstance(edge_data, dict) and 0 in edge_data:
            edge_data = edge_data[0]
        
        if edge_data and any(k in edge_data.get('description', '') for k in all_causal_keywords):
            # 2-hop 인과 체인 생성
            chain_nodes = direct_pattern['entities'] + [next_node]
            
            # 텍스트 증거 수집
            texts = []
            for node in chain_nodes:
                if node in G:
                    for text_info in G.nodes[node].get('original_texts', [])[:1]:
                        texts.append(text_info['text'])
            
            pattern = {
                'question_type': 'causal_chains',
                'pattern_subtype': '2hop_causal_chain',
                'chain_length': 2,
                'entities': chain_nodes,
                'entity_types': [G.nodes[n]['type'] for n in chain_nodes],
                'causal_chain': [
                    {
                        'from': chain_nodes[0],
                        'to': chain_nodes[1],
                        'description': direct_pattern['causal_relation']['description']
                    },
                    {
                        'from': chain_nodes[1],
                        'to': chain_nodes[2],
                        'description': edge_data['description']
                    }
                ],
                'source_texts': texts[:4],
                'pattern_template': f"{chain_nodes[0]}에서 시작된 연쇄적 영향이 {chain_nodes[2]}에 이르는 경로는?",
                'difficulty': 'medium'
            }
            
            causal_chain_patterns.append(pattern)
            
            # 3-hop으로 확장 시도
            for third_node in G.neighbors(next_node):
                if third_node not in chain_nodes:  # 사이클 방지
                    edge_data2 = G.get_edge_data(next_node, third_node)
                    if isinstance(edge_data2, dict) and 0 in edge_data2:
                        edge_data2 = edge_data2[0]
                    
                    if edge_data2 and any(k in edge_data2.get('description', '') for k in all_causal_keywords):
                        # 3-hop 인과 체인
                        extended_chain = chain_nodes + [third_node]
                        
                        texts_3hop = texts[:3]  # 기존 텍스트
                        if third_node in G:
                            for text_info in G.nodes[third_node].get('original_texts', [])[:1]:
                                texts_3hop.append(text_info['text'])
                        
                        pattern_3hop = {
                            'question_type': 'causal_chains',
                            'pattern_subtype': '3hop_causal_chain',
                            'chain_length': 3,
                            'entities': extended_chain,
                            'entity_types': [G.nodes[n]['type'] for n in extended_chain],
                            'causal_chain': pattern['causal_chain'] + [{
                                'from': extended_chain[2],
                                'to': extended_chain[3],
                                'description': edge_data2['description']
                            }],
                            'source_texts': texts_3hop[:5],
                            'pattern_template': f"{extended_chain[0]}의 영향이 {extended_chain[3]}까지 미치는 다단계 인과관계는?",
                            'difficulty': 'hard'
                        }
                        
                        causal_chain_patterns.append(pattern_3hop)
                        
                        if len(causal_chain_patterns) >= 40:
                            break
            
            if len(causal_chain_patterns) >= 40:
                break
    
    if len(causal_chain_patterns) >= 40:
        break

print(f"✓ 인과관계 체인 {len(causal_chain_patterns)}개 추출")

# 3. 복합 인과관계 패턴 찾기 (분기/수렴형)
print("\n3. 복합 인과관계 패턴 추출...")
complex_causal_patterns = []

# 3.1 분기형 인과관계 (한 원인이 여러 결과로)
print("   3.1 분기형 인과관계 찾기...")
for node in list(G.nodes())[:200]:  # 상위 200개 노드 검사
    if node not in G:
        continue
        
    # node에서 나가는 인과관계들
    causal_effects = []
    for neighbor in G.neighbors(node):
        edge_data = G.get_edge_data(node, neighbor)
        if isinstance(edge_data, dict) and 0 in edge_data:
            edge_data = edge_data[0]
            
        if edge_data and any(k in edge_data.get('description', '') for k in all_causal_keywords):
            causal_effects.append({
                'effect': neighbor,
                'type': G.nodes[neighbor]['type'],
                'description': edge_data['description'],
                'weight': edge_data.get('weight', 0)
            })
    
    if len(causal_effects) >= 2:  # 2개 이상의 결과
        # 텍스트 증거 수집
        texts = []
        if node in G:
            for text_info in G.nodes[node].get('original_texts', [])[:2]:
                texts.append(text_info['text'])
        
        for effect in causal_effects[:3]:
            if effect['effect'] in G:
                for text_info in G.nodes[effect['effect']].get('original_texts', [])[:1]:
                    texts.append(text_info['text'])
        
        pattern = {
            'question_type': 'causal_chains',
            'pattern_subtype': 'branching_causal',
            'cause_entity': node,
            'cause_type': G.nodes[node]['type'],
            'effects': causal_effects[:5],  # 최대 5개
            'effect_count': len(causal_effects),
            'source_texts': texts[:5],
            'pattern_template': f"{node}이(가) 야기하는 다양한 영향들은?",
            'difficulty': 'medium'
        }
        
        complex_causal_patterns.append(pattern)

# 3.2 수렴형 인과관계 (여러 원인이 한 결과로)
print("   3.2 수렴형 인과관계 찾기...")
for node in list(G.nodes())[:200]:  # 상위 200개 노드 검사
    if node not in G:
        continue
        
    # node로 들어오는 인과관계들
    causal_causes = []
    for predecessor in G.predecessors(node):
        edge_data = G.get_edge_data(predecessor, node)
        if isinstance(edge_data, dict) and 0 in edge_data:
            edge_data = edge_data[0]
            
        if edge_data and any(k in edge_data.get('description', '') for k in all_causal_keywords):
            causal_causes.append({
                'cause': predecessor,
                'type': G.nodes[predecessor]['type'],
                'description': edge_data['description'],
                'weight': edge_data.get('weight', 0)
            })
    
    if len(causal_causes) >= 2:  # 2개 이상의 원인
        # 텍스트 증거 수집
        texts = []
        for cause in causal_causes[:3]:
            if cause['cause'] in G:
                for text_info in G.nodes[cause['cause']].get('original_texts', [])[:1]:
                    texts.append(text_info['text'])
        
        if node in G:
            for text_info in G.nodes[node].get('original_texts', [])[:2]:
                texts.append(text_info['text'])
        
        pattern = {
            'question_type': 'causal_chains',
            'pattern_subtype': 'converging_causal',
            'effect_entity': node,
            'effect_type': G.nodes[node]['type'],
            'causes': causal_causes[:5],  # 최대 5개
            'cause_count': len(causal_causes),
            'source_texts': texts[:5],
            'pattern_template': f"{node}에 영향을 미치는 다양한 요인들은?",
            'difficulty': 'medium'
        }
        
        complex_causal_patterns.append(pattern)

# 복합 패턴 중 품질 높은 것만 선택
complex_causal_patterns.sort(key=lambda x: x.get('effect_count', x.get('cause_count', 0)), reverse=True)
complex_causal_patterns = complex_causal_patterns[:10]  # 상위 10개

print(f"✓ 복합 인과관계 {len(complex_causal_patterns)}개 추출")

# 4. 모든 인과관계 패턴 통합
causal_chains_patterns = direct_causal_patterns + causal_chain_patterns + complex_causal_patterns

print(f"\n4. 총 {len(causal_chains_patterns)}개 causal chains 패턴 추출")
print(f"   - 직접적 인과관계: {len(direct_causal_patterns)}개")
print(f"   - 인과관계 체인: {len(causal_chain_patterns)}개")
print(f"   - 복합 인과관계: {len(complex_causal_patterns)}개")

# 5. 샘플 패턴 출력
print("\n5. 샘플 Causal Chains 패턴:")

# 직접적 인과관계 샘플
if direct_causal_patterns:
    pattern = direct_causal_patterns[0]
    print(f"\n   [직접적 인과관계]")
    print(f"   {pattern['entities'][0]} → {pattern['entities'][1]}")
    print(f"   타입: {pattern['entity_types'][0]} → {pattern['entity_types'][1]}")
    print(f"   관계: {pattern['causal_relation']['description'][:60]}...")
    print(f"   가중치: {pattern['causal_relation']['weight']}")

# 인과관계 체인 샘플
chain_2hop = [p for p in causal_chain_patterns if p['chain_length'] == 2]
chain_3hop = [p for p in causal_chain_patterns if p['chain_length'] == 3]

if chain_2hop:
    pattern = chain_2hop[0]
    print(f"\n   [2-hop 인과 체인]")
    print(f"   경로: {' → '.join(pattern['entities'])}")
    print(f"   타입: {' → '.join(pattern['entity_types'])}")

if chain_3hop:
    pattern = chain_3hop[0]
    print(f"\n   [3-hop 인과 체인]")
    print(f"   경로: {' → '.join(pattern['entities'])}")
    print(f"   타입: {' → '.join(pattern['entity_types'])}")

# 복합 인과관계 샘플
branching = [p for p in complex_causal_patterns if p['pattern_subtype'] == 'branching_causal']
converging = [p for p in complex_causal_patterns if p['pattern_subtype'] == 'converging_causal']

if branching:
    pattern = branching[0]
    print(f"\n   [분기형 인과관계]")
    print(f"   원인: {pattern['cause_entity']} ({pattern['cause_type']})")
    print(f"   결과 수: {pattern['effect_count']}개")
    for i, effect in enumerate(pattern['effects'][:3], 1):
        print(f"     {i}. → {effect['effect']} ({effect['type']})")

if converging:
    pattern = converging[0]
    print(f"\n   [수렴형 인과관계]")
    print(f"   결과: {pattern['effect_entity']} ({pattern['effect_type']})")
    print(f"   원인 수: {pattern['cause_count']}개")
    for i, cause in enumerate(pattern['causes'][:3], 1):
        print(f"     {i}. {cause['cause']} ({cause['type']}) →")

print(f"\n✓ Causal chains 패턴 추출 완료!")

### Cell 29: Causal Chains 패턴 상세 분석

추출된 인과관계 체인 패턴들의 구조와 특성을 상세히 분석합니다.

In [ ]:
print("=== Causal Chains 패턴 상세 분석 ===\n")

print("1. 기본 통계:")
print(f"   총 {len(causal_chains_patterns)}개의 causal chains 패턴")

# 패턴 타입별 분포
pattern_dist = defaultdict(int)
for pattern in causal_chains_patterns:
    pattern_dist[pattern['pattern_subtype']] += 1

print("\n   패턴 타입별 분포:")
for ptype, count in pattern_dist.items():
    percentage = count / len(causal_chains_patterns) * 100
    print(f"   - {ptype}: {count}개 ({percentage:.1f}%)")

# 1. 직접적 인과관계 분석
print("\n\n2. 직접적 인과관계 분석:")

if direct_causal_patterns:
    # 인과관계 타입별 분류
    causal_type_dist = defaultdict(int)
    causal_type_examples = defaultdict(list)
    
    for pattern in direct_causal_patterns:
        ctype = pattern['causal_relation']['type']
        causal_type_dist[ctype] += 1
        if len(causal_type_examples[ctype]) < 3:
            causal_type_examples[ctype].append(pattern)
    
    print("\n   2.1. 인과관계 타입 분포:")
    for ctype, count in sorted(causal_type_dist.items(), key=lambda x: x[1], reverse=True):
        print(f"       {ctype}: {count}개")
    
    # 가중치 분석
    weights = [p['causal_relation']['weight'] for p in direct_causal_patterns]
    print(f"\n   2.2. 인과관계 강도 (가중치) 분석:")
    print(f"       평균: {np.mean(weights):.1f}")
    print(f"       중앙값: {np.median(weights):.1f}")
    print(f"       최대: {max(weights)}")
    print(f"       상위 25%: {np.percentile(weights, 75):.1f}")
    
    # 엔티티 타입 변환 패턴
    type_transitions = defaultdict(int)
    for pattern in direct_causal_patterns:
        transition = f"{pattern['entity_types'][0]} → {pattern['entity_types'][1]}"
        type_transitions[transition] += 1
    
    print("\n   2.3. 주요 인과관계 타입 변환:")
    for transition, count in sorted(type_transitions.items(), 
                                   key=lambda x: x[1], reverse=True)[:5]:
        print(f"       {transition}: {count}회")
    
    # 인과관계 예시
    print("\n   2.4. 인과관계 타입별 예시:")
    for ctype, examples in causal_type_examples.items():
        if examples:
            print(f"\n       [{ctype}]")
            example = examples[0]
            print(f"       {example['entities'][0]} → {example['entities'][1]}")
            print(f"       설명: {example['causal_relation']['description'][:80]}...")

# 2. 인과관계 체인 분석
print("\n\n3. 인과관계 체인 분석:")

if causal_chain_patterns:
    # 체인 길이별 분포
    chain_length_dist = defaultdict(int)
    for pattern in causal_chain_patterns:
        chain_length_dist[pattern['chain_length']] += 1
    
    print("\n   3.1. 체인 길이별 분포:")
    for length, count in sorted(chain_length_dist.items()):
        print(f"       {length}-hop: {count}개")
    
    # 2-hop 체인 분석
    hop2_patterns = [p for p in causal_chain_patterns if p['chain_length'] == 2]
    if hop2_patterns:
        print("\n   3.2. 2-hop 인과 체인 분석:")
        print(f"       총 {len(hop2_patterns)}개")
        
        # 중간 노드 역할 분석
        middle_node_types = defaultdict(int)
        for pattern in hop2_patterns:
            middle_type = pattern['entity_types'][1]
            middle_node_types[middle_type] += 1
        
        print("\n       중간 매개 노드 타입:")
        for ntype, count in sorted(middle_node_types.items(), 
                                  key=lambda x: x[1], reverse=True)[:5]:
            print(f"       - {ntype}: {count}회")
        
        # 샘플 2-hop 체인
        if hop2_patterns:
            pattern = hop2_patterns[0]
            print("\n       2-hop 체인 예시:")
            print(f"       경로: {' → '.join(pattern['entities'])}")
            for i, link in enumerate(pattern['causal_chain']):
                print(f"       단계 {i+1}: {link['description'][:60]}...")
    
    # 3-hop 체인 분석
    hop3_patterns = [p for p in causal_chain_patterns if p['chain_length'] == 3]
    if hop3_patterns:
        print("\n   3.3. 3-hop 인과 체인 분석:")
        print(f"       총 {len(hop3_patterns)}개")
        
        # 엔티티 타입 다양성
        type_diversities = []
        for pattern in hop3_patterns:
            unique_types = len(set(pattern['entity_types']))
            type_diversities.append(unique_types)
        
        print(f"       평균 타입 다양성: {np.mean(type_diversities):.1f}개 고유 타입")
        
        # 샘플 3-hop 체인
        if hop3_patterns:
            pattern = hop3_patterns[0]
            print("\n       3-hop 체인 예시:")
            print(f"       경로: {' → '.join(pattern['entities'][:2])}...{pattern['entities'][-1]}")
            print(f"       타입 체인: {' → '.join(pattern['entity_types'])}")

# 3. 복합 인과관계 분석
print("\n\n4. 복합 인과관계 패턴 분석:")

branching_patterns = [p for p in complex_causal_patterns 
                     if p['pattern_subtype'] == 'branching_causal']
converging_patterns = [p for p in complex_causal_patterns 
                      if p['pattern_subtype'] == 'converging_causal']

if branching_patterns:
    print("\n   4.1. 분기형 인과관계:")
    print(f"       총 {len(branching_patterns)}개 패턴")
    
    # 평균 결과 수
    effect_counts = [p['effect_count'] for p in branching_patterns]
    print(f"       평균 결과 수: {np.mean(effect_counts):.1f}개")
    print(f"       최대 결과 수: {max(effect_counts)}개")
    
    # 주요 원인 노드
    print("\n       주요 원인 노드 (다중 영향):")
    for i, pattern in enumerate(branching_patterns[:3], 1):
        print(f"       {i}. {pattern['cause_entity']} ({pattern['cause_type']})")
        print(f"          → {pattern['effect_count']}개 결과 야기")

if converging_patterns:
    print("\n   4.2. 수렴형 인과관계:")
    print(f"       총 {len(converging_patterns)}개 패턴")
    
    # 평균 원인 수
    cause_counts = [p['cause_count'] for p in converging_patterns]
    print(f"       평균 원인 수: {np.mean(cause_counts):.1f}개")
    print(f"       최대 원인 수: {max(cause_counts)}개")
    
    # 주요 결과 노드
    print("\n       주요 결과 노드 (다중 원인):")
    for i, pattern in enumerate(converging_patterns[:3], 1):
        print(f"       {i}. {pattern['effect_entity']} ({pattern['effect_type']})")
        print(f"          ← {pattern['cause_count']}개 원인에 의해 영향받음")

# 4. 인과관계 강도 평가
print("\n\n5. 인과관계 패턴 품질 평가:")

causal_quality_scores = []
high_quality_causal = []

for pattern in causal_chains_patterns:
    score = 0
    
    # 직접적 인과관계
    if pattern['pattern_subtype'] == 'direct_causal':
        # 가중치 기반 점수
        weight = pattern['causal_relation']['weight']
        if weight >= 50:
            score += 3
        elif weight >= 20:
            score += 2
        elif weight >= 10:
            score += 1
        
        # 인과관계 타입
        if pattern['causal_relation']['type'] != 'general':
            score += 1
    
    # 인과 체인
    elif pattern['pattern_subtype'] in ['2hop_causal_chain', '3hop_causal_chain']:
        score += pattern['chain_length']  # 체인 길이에 비례
        
        # 타입 다양성
        unique_types = len(set(pattern['entity_types']))
        score += unique_types - 1
    
    # 복합 인과관계
    else:
        if pattern['pattern_subtype'] == 'branching_causal':
            score += min(pattern['effect_count'], 5)
        elif pattern['pattern_subtype'] == 'converging_causal':
            score += min(pattern['cause_count'], 5)
    
    # 텍스트 증거
    if len(pattern['source_texts']) >= 3:
        score += 1
    
    pattern['causal_quality'] = score
    causal_quality_scores.append(score)
    
    if score >= 5:
        high_quality_causal.append(pattern)

print(f"   고품질 인과관계: {len(high_quality_causal)}개 / {len(causal_chains_patterns)}개")
print(f"   평균 품질 점수: {np.mean(causal_quality_scores):.2f}")

# 5. 최고 품질 인과관계 패턴 예시
print("\n\n6. 최고 품질 인과관계 패턴 예시:")
top_causal = sorted(causal_chains_patterns, 
                   key=lambda x: x.get('causal_quality', 0), 
                   reverse=True)[:5]

top_quality_causal = []
for i, pattern in enumerate(top_causal, 1):
    print(f"\n   예시 {i} (품질 점수: {pattern.get('causal_quality', 0)}):")
    print(f"   패턴 타입: {pattern['pattern_subtype']}")
    
    if pattern['pattern_subtype'] == 'direct_causal':
        print(f"   인과관계: {pattern['entities'][0]} → {pattern['entities'][1]}")
        print(f"   관계 타입: {pattern['causal_relation']['type']}")
        print(f"   가중치: {pattern['causal_relation']['weight']}")
        print(f"   설명: {pattern['causal_relation']['description'][:80]}...")
        
        top_quality_causal.append({
            'pattern_subtype': pattern['pattern_subtype'],
            'entities': pattern['entities'],
            'causal_relation': pattern['causal_relation'],
            'quality_score': pattern.get('causal_quality', 0)
        })
        
    elif pattern['pattern_subtype'] in ['2hop_causal_chain', '3hop_causal_chain']:
        print(f"   체인 경로: {' → '.join(pattern['entities'])}")
        print(f"   체인 길이: {pattern['chain_length']}-hop")
        print(f"   타입 다양성: {len(set(pattern['entity_types']))}개 고유 타입")
        
        top_quality_causal.append({
            'pattern_subtype': pattern['pattern_subtype'],
            'chain_length': pattern['chain_length'],
            'entities': pattern['entities'],
            'entity_types': pattern['entity_types'],
            'quality_score': pattern.get('causal_quality', 0)
        })
        
    else:  # 복합 인과관계
        if pattern['pattern_subtype'] == 'branching_causal':
            print(f"   원인: {pattern['cause_entity']}")
            print(f"   결과 수: {pattern['effect_count']}개")
            effects_sample = [e['effect'] for e in pattern['effects'][:3]]
            print(f"   주요 결과: {', '.join(effects_sample)}")
        else:  # converging_causal
            print(f"   결과: {pattern['effect_entity']}")
            print(f"   원인 수: {pattern['cause_count']}개")
            causes_sample = [c['cause'] for c in pattern['causes'][:3]]
            print(f"   주요 원인: {', '.join(causes_sample)}")
        
        top_quality_causal.append({
            'pattern_subtype': pattern['pattern_subtype'],
            'quality_score': pattern.get('causal_quality', 0),
            'entity_info': {
                'cause_entity': pattern.get('cause_entity'),
                'effect_entity': pattern.get('effect_entity'),
                'count': pattern.get('effect_count', pattern.get('cause_count', 0))
            }
        })

# 6. GraphRAG의 인과관계 탐지 장점
print("\n\n7. GraphRAG의 인과관계 탐지 장점:")
print("\n   전통적 RAG의 한계:")
print("   - 단일 문서 내 명시적 인과관계만 탐지")
print("   - 다단계 인과 체인 추적 불가")
print("   - 복합적 인과관계 파악 어려움")
print("\n   GraphRAG의 강점:")
print("   - 그래프 구조로 다단계 인과 체인 추적")
print("   - 엣지 가중치로 인과관계 강도 평가")
print("   - 분기/수렴형 복합 인과관계 발견")
print("   - 문서 간 걸친 인과관계 연결")

# 분석 결과 저장
analysis_results_causal = {
    'pattern_count': len(causal_chains_patterns),
    'pattern_distribution': dict(pattern_dist),
    'direct_causal_analysis': {
        'count': len(direct_causal_patterns),
        'causal_type_distribution': dict(causal_type_dist),
        'weight_statistics': {
            'mean': float(np.mean(weights)) if weights else 0,
            'median': float(np.median(weights)) if weights else 0,
            'max': float(max(weights)) if weights else 0
        },
        'type_transitions': [(k, v) for k, v in sorted(type_transitions.items(), 
                                                       key=lambda x: x[1], reverse=True)[:10]]
    },
    'chain_analysis': {
        'chain_length_distribution': dict(chain_length_dist),
        '2hop_count': len(hop2_patterns),
        '3hop_count': len(hop3_patterns),
        'middle_node_types': dict(middle_node_types) if 'middle_node_types' in locals() else {}
    },
    'complex_causal_analysis': {
        'branching_count': len(branching_patterns),
        'converging_count': len(converging_patterns),
        'avg_branch_effects': float(np.mean(effect_counts)) if effect_counts else 0,
        'avg_converge_causes': float(np.mean(cause_counts)) if cause_counts else 0
    },
    'quality_analysis': {
        'high_quality_count': len(high_quality_causal),
        'average_quality': float(np.mean(causal_quality_scores)),
        'quality_distribution': pd.Series(causal_quality_scores).value_counts().to_dict()
    },
    'top_quality_examples': top_quality_causal[:5]
}

# JSON으로 저장
output_dir_causal = output_path / 'causal_chains'
output_dir_causal.mkdir(exist_ok=True)

with open(output_dir_causal / 'causal_pattern_analysis.json', 'w', encoding='utf-8') as f:
    json.dump(analysis_results_causal, f, ensure_ascii=False, indent=2)

# 패턴 자체도 저장
with open(output_dir_causal / 'causal_patterns.json', 'w', encoding='utf-8') as f:
    patterns_to_save = []
    for p in causal_chains_patterns:
        pattern_copy = {
            'question_type': p['question_type'],
            'pattern_subtype': p['pattern_subtype'],
            'source_texts': p['source_texts'][:5],
            'pattern_template': p['pattern_template'],
            'difficulty': p['difficulty'],
            'causal_quality': p.get('causal_quality', 0)
        }
        
        # 서브타입별 추가 정보
        if p['pattern_subtype'] == 'direct_causal':
            pattern_copy.update({
                'chain_length': p['chain_length'],
                'entities': p['entities'],
                'entity_types': p['entity_types'],
                'causal_relation': p['causal_relation']
            })
        elif p['pattern_subtype'] in ['2hop_causal_chain', '3hop_causal_chain']:
            pattern_copy.update({
                'chain_length': p['chain_length'],
                'entities': p['entities'],
                'entity_types': p['entity_types'],
                'causal_chain': p['causal_chain']
            })
        elif p['pattern_subtype'] == 'branching_causal':
            pattern_copy.update({
                'cause_entity': p['cause_entity'],
                'cause_type': p['cause_type'],
                'effects': p['effects'][:10],
                'effect_count': p['effect_count']
            })
        elif p['pattern_subtype'] == 'converging_causal':
            pattern_copy.update({
                'effect_entity': p['effect_entity'],
                'effect_type': p['effect_type'],
                'causes': p['causes'][:10],
                'cause_count': p['cause_count']
            })
        
        patterns_to_save.append(pattern_copy)
    
    json.dump(patterns_to_save, f, ensure_ascii=False, indent=2)

print(f"\n✓ Causal chains 패턴 분석 결과 저장 완료:")
print(f"   - {output_dir_causal / 'causal_pattern_analysis.json'}")
print(f"   - {output_dir_causal / 'causal_patterns.json'}")
print("\n✓ Causal Chains 패턴 상세 분석 완료!")
print("\n✓ Phase 3 완료! 모든 QA 패턴 추출 및 분석이 완료되었습니다.")

## Phase 3 최종 요약

Phase 3에서 성공적으로 5가지 유형의 QA 패턴을 추출했습니다:

| QA 유형 | 추출된 패턴 수 | 비율 |
|---------|---------------|------|
| Multi-hop Reasoning | 100개 | 24.6% |
| Community Synthesis | 38개 | 9.3% |
| Cross-context Integration | 100개 | 24.6% |
| Implicit Relationships | 69개 | 16.9% |
| Causal Chains | 100개 | 24.6% |
| **총계** | **407개** | **100%** |

각 패턴 유형의 상세 분석은 `qa_types.md` 파일에 문서화되었습니다.

## Phase 4 질문 생성

### Cell 30: Phase 4 설정 - 난이도별 질문 템플릿 (Easy/Hard 이원화)

In [ ]:
# Phase 4 설정 - 난이도별 질문 템플릿 (Easy/Hard 이원화)
import openai
from typing import List, Dict, Any
import random

# 난이도별 질문 생성 템플릿 (Easy/Hard 이원화, 구조적 답변 유도)
QUESTION_TEMPLATES_BY_DIFFICULTY = {
    'multi_hop': {
        'easy': {
            '2hop': [
                "{start}과(와) {end}는 {middle}을(를) 통해 어떻게 연결되나요?",
                "{start}에서 {middle}을(를) 거쳐 {end}로 이어지는 관계를 설명하세요.",
                "{middle}이(가) {start}과(와) {end} 사이에서 하는 역할은 무엇인가요?"
            ],
            '3hop': [
                "{start}이(가) {middle1}, {middle2}를 거쳐 {end}에 이르는 과정을 설명하세요.",
                "{start}에서 출발하여 {end}까지 도달하는 경로 ({middle1}, {middle2} 경유)를 분석하세요."
            ]
        },
        'hard': {
            '2hop': [
                "{start}과(와) {end} 사이의 연결 경로를 찾아 설명하세요. 경로를 (노드1 → 노드2 → 노드3) 형태로 제시하세요.",
                "그래프에서 {start}과(와) {end}를 연결하는 경로를 탐색하고 각 단계를 명시하세요. 답변 형식: [시작] → [중간] → [끝]",
                "{start}이(가) {end}에 미치는 영향 경로를 추적하세요. 모든 중간 단계를 → 로 연결하여 표시하세요."
            ],
            '3hop': [
                "{start}에서 {end}로 이어지는 모든 중간 단계를 찾아 순서대로 나열하세요. 형식: A → B → C → D",
                "네트워크 상에서 {start}과(와) {end}의 연결 경로를 도출하세요. 각 노드를 → 로 연결하여 표시하세요.",
                "{start}과(와) {end} 사이의 다단계 경로를 분석하세요. 전체 경로를 노드1 → 노드2 → ... → 노드N 형식으로 제시하세요."
            ]
        }
    },
    
    'community_synthesis': {
        'easy': [
            "{community_title} 커뮤니티의 주요 구성원 {entity1}, {entity2}의 역할을 설명하세요.",
            "{community_title}에서 {entity1}과(와) {entity2}는 어떤 관계인가요?",
            "{community_title} 커뮤니티의 핵심 주제와 주요 엔티티들을 소개하세요."
        ],
        'hard': [
            "{community_title} 커뮤니티에서 중심성 상위 5개 엔티티를 제시하고, 주요 관계 유형 3개를 빈도순으로 나열하세요.",
            "{community_title}의 핵심 구성원 상위 5개와 대표적인 관계 3개를 (주체, 관계, 대상) 형태로 제시하세요.",
            "{community_title} 내부의 주요 노드 5개를 중심성 기준으로 나열하고, 가장 빈번한 관계 타입 3개를 제시하세요."
        ]
    },
    
    'cross_context': {
        'easy': [
            "{entity}이(가) {context_count}개 문맥에서 언급된 내용을 요약하세요.",
            "{entity}과(와) {related_entities} 사이의 관계를 여러 문맥에서 찾아 정리하세요.",
            "다양한 문서에서 언급된 {entity}의 정보를 종합하여 설명하세요."
        ],
        'hard': [
            "{entity}와 관련된 정보를 분석하세요. 서로 다른 2개 이상의 문맥에서 나온 정보를 결합하고, 공통점 2개와 차이점 1개를 제시하세요.",
            "다중 문서에 걸친 {entity} 정보를 통합하세요. 최소 2개의 서로 다른 문맥 근거를 명시하고 통합된 이해를 제시하세요.",
            "{entity}에 대해 최소 3개의 서로 다른 문맥에서 정보를 추출하고, 각 문맥의 고유한 정보를 구분하여 제시하세요."
        ]
    },
    
    'implicit_relationships': {
        'easy': [
            "{entity1}과(와) {entity2}가 {community}에서 공유하는 연결점은 무엇인가요?",
            "{community} 내에서 {entity1}과(와) {entity2}의 간접적 관련성을 찾아보세요.",
            "{entity1}과(와) {entity2}가 {common_neighbors}을(를) 통해 어떻게 연결되나요?",
            "{common_count}개의 공통 연결을 가진 {entity1}과(와) {entity2}의 관계를 분석하세요."
        ],
        'hard': [
            "{entity1}과(와) {entity2}의 암시적 관계를 분석하세요. 공통 매개 엔티티 또는 2-hop 경로를 1개 이상 명시하세요.",
            "직접 연결되지 않은 {entity1}과(와) {entity2} 사이의 간접 경로를 탐색하세요. 매개 노드를 명확히 제시하세요.",
            "{entity1}과(와) {entity2}의 숨겨진 연결을 찾으세요. 중간 매개체와 함께 경로를 (A → 매개 → B) 형식으로 제시하세요."
        ]
    },
    
    'causal_chains': {
        'easy': [
            "{cause}이(가) {effect}를 일으키는 과정을 설명하세요.",
            "{cause}와(과) {effect}의 인과관계를 분석하세요.",
            "{start}이(가) {middle}을(를) 거쳐 {end}에 이르는 인과 과정을 설명하세요.",
            "{start} → {middle} → {end}의 연쇄적 인과관계를 추적하세요."
        ],
        'hard': [
            "인과관계 체인을 발견하고 각 단계를 (원인 → 결과) 형태로 순서대로 제시하세요.",
            "복잡한 인과 네트워크에서 주요 경로를 찾아 A → B → C 형식으로 나열하세요.",
            "{start}에서 {end}까지의 인과 경로를 추적하고, 모든 중간 단계를 → 로 연결하여 명시하세요.",
            "다중 인과관계를 분석하고, 각 인과 경로를 명확히 구분하여 화살표(→)로 표시하세요."
        ]
    }
}

# 난이도 결정 함수 (Easy/Hard 이원화)
def determine_difficulty(pattern: Dict, qa_type: str) -> str:
    """패턴의 복잡도에 따라 난이도 자동 결정 (Easy 35%, Hard 65%)"""
    
    # 전체적으로 Hard 비율을 65%로 설정
    base_hard_probability = 0.65
    
    if qa_type == 'multi_hop':
        path_length = len(pattern.get('path', []))
        if path_length <= 3:
            # 2-hop은 easy 확률 조금 높게
            return 'hard' if random.random() < 0.6 else 'easy'
        else:
            # 3-hop 이상은 hard 확률 높게
            return 'hard' if random.random() < 0.8 else 'easy'
            
    elif qa_type == 'community_synthesis':
        entity_count = pattern.get('entity_count', 0)
        if entity_count < 15:
            return 'hard' if random.random() < 0.6 else 'easy'
        else:
            # 큰 커뮤니티는 hard 위주
            return 'hard' if random.random() < 0.75 else 'easy'
            
    elif qa_type == 'cross_context':
        context_count = len(pattern.get('text_units', []))
        if context_count <= 3:
            return 'hard' if random.random() < 0.5 else 'easy'
        else:
            # 많은 컨텍스트는 hard 위주
            return 'hard' if random.random() < 0.7 else 'easy'
            
    elif qa_type == 'implicit_relationships':
        # implicit는 본질적으로 어려우므로 hard 비율 높게
        return 'hard' if random.random() < 0.7 else 'easy'
            
    elif qa_type == 'causal_chains':
        if 'chain' in pattern and len(pattern['chain']) > 2:
            # 긴 체인은 hard 위주
            return 'hard' if random.random() < 0.75 else 'easy'
        else:
            return 'hard' if random.random() < 0.6 else 'easy'
    
    # 기본값
    return 'hard' if random.random() < base_hard_probability else 'easy'

# 템플릿 선택 헬퍼 함수
def get_templates_for_pattern(pattern: Dict, qa_type: str, difficulty: str) -> List[str]:
    """패턴에 맞는 템플릿 가져오기"""
    templates = QUESTION_TEMPLATES_BY_DIFFICULTY.get(qa_type, {})
    
    if qa_type == 'multi_hop':
        path_length = len(pattern.get('path', []))
        hop_type = '2hop' if path_length == 3 else '3hop'
        if isinstance(templates.get(difficulty), dict):
            return templates[difficulty].get(hop_type, [])
        else:
            return templates.get(difficulty, [])
    
    else:
        return templates.get(difficulty, [])

print("Easy/Hard 이원화된 난이도별 질문 템플릿 설정 완료")

## Cell 31: 템플릿 기반 질문 생성 함수 생성
패턴 정보를 이용해서 질문 템플릿을 채우고, 난이도별 질문을 생성합니다

In [ ]:
# 템플릿 기반 질문 생성 함수
def fill_question_template(template: str, pattern: Dict, qa_type: str) -> str:
    """패턴 정보로 질문 템플릿 채우기"""
    filled = template
    
    if qa_type == 'multi_hop':
        if 'path' in pattern:
            path = pattern['path']
            filled = filled.replace('{start}', path[0])
            filled = filled.replace('{end}', path[-1])
            
            if len(path) == 3:
                filled = filled.replace('{middle}', path[1])
            elif len(path) == 4:
                filled = filled.replace('{middle1}', path[1])
                filled = filled.replace('{middle2}', path[2])
            elif len(path) > 4:
                # 4개 초과 노드는 중간 노드들 표시
                middles = path[1:-1]
                filled = filled.replace('{middle}', ', '.join(middles))
    
    elif qa_type == 'community_synthesis':
        filled = filled.replace('{community_title}', pattern.get('community_title', ''))
        if 'entities' in pattern and len(pattern['entities']) >= 2:
            filled = filled.replace('{entity1}', pattern['entities'][0])
            filled = filled.replace('{entity2}', pattern['entities'][1])
    
    elif qa_type == 'cross_context':
        filled = filled.replace('{entity}', pattern.get('anchor_entity', ''))
        filled = filled.replace('{context_count}', str(pattern.get('unit_count', 0)))
        if 'related_entities' in pattern:
            # related_entities가 딕셔너리 리스트인 경우
            if pattern['related_entities'] and isinstance(pattern['related_entities'][0], dict):
                related = [e['entity'] for e in pattern['related_entities'][:3]]
            else:
                related = pattern['related_entities'][:3]
            filled = filled.replace('{related_entities}', ', '.join(related))
    
    elif qa_type == 'implicit_relationships':
        if 'entities' in pattern and len(pattern['entities']) >= 2:
            filled = filled.replace('{entity1}', pattern['entities'][0])
            filled = filled.replace('{entity2}', pattern['entities'][1])
        
        # 커뮤니티 정보
        if pattern.get('pattern_subtype') == 'community_based':
            filled = filled.replace('{community}', pattern.get('community_title', ''))
        
        # 공통 이웃 정보
        if 'common_neighbor_count' in pattern:
            filled = filled.replace('{common_count}', str(pattern['common_neighbor_count']))
        
        if 'common_neighbors' in pattern and pattern['common_neighbors']:
            neighbors = pattern['common_neighbors'][:3]
            filled = filled.replace('{common_neighbors}', ', '.join(neighbors))
    
    elif qa_type == 'causal_chains':
        # 직접 인과관계
        if pattern.get('pattern_subtype') == 'direct_causal':
            filled = filled.replace('{cause}', pattern['entities'][0])
            filled = filled.replace('{effect}', pattern['entities'][1])
        
        # 인과 체인
        elif pattern.get('pattern_subtype') in ['2hop_causal_chain', '3hop_causal_chain']:
            if 'entities' in pattern and len(pattern['entities']) >= 2:
                filled = filled.replace('{start}', pattern['entities'][0])
                filled = filled.replace('{end}', pattern['entities'][-1])
                if len(pattern['entities']) >= 3:
                    filled = filled.replace('{middle}', pattern['entities'][1])
        
        # 복합 인과관계
        elif pattern.get('pattern_subtype') == 'branching_causal':
            filled = filled.replace('{cause}', pattern.get('cause_entity', ''))
            if 'effects' in pattern and pattern['effects']:
                effect_entities = [e['effect'] for e in pattern['effects'][:3]]
                filled = filled.replace('{effects}', ', '.join(effect_entities))
        
        elif pattern.get('pattern_subtype') == 'converging_causal':
            filled = filled.replace('{effect}', pattern.get('effect_entity', ''))
            if 'causes' in pattern and pattern['causes']:
                cause_entities = [c['cause'] for c in pattern['causes'][:3]]
                filled = filled.replace('{causes}', ', '.join(cause_entities))
    
    return filled

def generate_questions_from_patterns(patterns: List[Dict], qa_type: str) -> List[Dict]:
    """패턴에서 난이도별 질문 생성"""
    questions = []
    
    for i, pattern in enumerate(patterns):
        # 난이도 자동 결정
        difficulty = determine_difficulty(pattern, qa_type)
        
        # 적절한 템플릿 선택
        templates = get_templates_for_pattern(pattern, qa_type, difficulty)
        
        if not templates:
            print(f"Warning: No templates found for {qa_type} pattern {i}")
            continue
        
        # 템플릿에서 질문 생성 (최대 2개)
        selected_templates = random.sample(templates, min(2, len(templates)))
        
        for j, template in enumerate(selected_templates):
            question = fill_question_template(template, pattern, qa_type)
            
            # 템플릿이 제대로 채워졌는지 확인
            if '{' not in question:  # 모든 placeholder가 채워짐
                questions.append({
                    'question_id': f"{qa_type}_{i}_{j}",
                    'pattern_id': f"{qa_type}_{i}",
                    'question': question,
                    'qa_type': qa_type,
                    'difficulty': difficulty,
                    'pattern': pattern  # 나중에 GT 생성용
                })
    
    return questions

print("템플릿 기반 질문 생성 함수 정의 완료")

## Cell 32: Phase 3 패턴 파일 로드
Phase 3에서 저장한 패턴 파일을 로드하여 qa 생성 기반을 마련합니다

In [ ]:
# Phase 3에서 저장된 패턴 파일 로드
import json

print("=== Phase 3 패턴 로드 중... ===")

# 패턴 파일 경로
pattern_files = {
    'multi_hop': [
        'multi_hop/2hop_patterns.json',
        'multi_hop/3hop_patterns.json'
    ],
    'community_synthesis': ['community_synthesis/community_patterns.json'],
    'cross_context': ['cross_context/cross_context_patterns.json'],
    'implicit_relationships': ['implicit_relationships/implicit_patterns.json'],
    'causal_chains': ['causal_chains/causal_patterns.json']
}

# 모든 패턴 로드
all_patterns = {}
for qa_type, files in pattern_files.items():
    patterns = []
    for file in files:
        file_path = output_path / file
        if file_path.exists():
            with open(file_path, 'r', encoding='utf-8') as f:
                file_patterns = json.load(f)
                patterns.extend(file_patterns)
                print(f"✓ {file}: {len(file_patterns)}개 패턴 로드")
        else:
            print(f"✗ {file}: 파일을 찾을 수 없음")
    
    all_patterns[qa_type] = patterns
    print(f"{qa_type}: 총 {len(patterns)}개 패턴\n")

# 총 패턴 수
total_patterns = sum(len(patterns) for patterns in all_patterns.values())
print(f"\n총 {total_patterns}개 패턴 로드 완료")

## Cell 33: QA 생성
각 QA 유형별로 질문을 생성합니다

In [ ]:
# 각 QA 유형별로 질문 생성
print("=== 질문 생성 시작 ===")

all_questions = {}
total_questions = 0

for qa_type, patterns in all_patterns.items():
    print(f"\n{qa_type} 질문 생성 중...")
    
    # 질문 생성
    questions = generate_questions_from_patterns(patterns, qa_type)
    
    # 난이도별 통계
    difficulty_counts = {'easy': 0, 'hard': 0}
    for q in questions:
        difficulty_counts[q['difficulty']] += 1
    
    print(f"  생성된 질문: {len(questions)}개")
    print(f"  - Easy: {difficulty_counts['easy']}개 ({difficulty_counts['easy']/len(questions)*100:.1f}%)")
    print(f"  - Hard: {difficulty_counts['hard']}개 ({difficulty_counts['hard']/len(questions)*100:.1f}%)")
    
    all_questions[qa_type] = questions
    total_questions += len(questions)

print(f"\n✓ 총 {total_questions}개 질문 생성 완료!")

# 샘플 질문 출력
print("\n=== 샘플 질문 (각 유형별 2개씩) ===")
for qa_type, questions in all_questions.items():
    print(f"\n[{qa_type}]")
    for q in questions[:2]:
        print(f"  {q['difficulty'].upper()}: {q['question']}")
        print(f"  ID: {q['question_id']}")

## Cell 34: 생성된 질문 저장
생성된 질문들을 파일로 저장합니다

In [ ]:
# 생성된 질문들을 파일로 저장
print("\n=== 생성된 질문 저장 중... ===")

# 각 QA 유형별로 저장
for qa_type, questions in all_questions.items():
    # 디렉토리 생성
    qa_dir = output_path / qa_type
    qa_dir.mkdir(exist_ok=True)
    
    # 질문 저장 (pattern 정보는 별도로 저장)
    questions_to_save = []
    for q in questions:
        q_copy = q.copy()
        # pattern 정보는 제외하고 저장 (너무 크므로)
        q_copy.pop('pattern', None)
        questions_to_save.append(q_copy)
    
    # JSON으로 저장
    output_file = qa_dir / 'generated_questions.json'
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(questions_to_save, f, ensure_ascii=False, indent=2)
    
    print(f"✓ {qa_type}: {output_file}")
    
    # 질문 분석 정보도 저장
    analysis = {
        'total_questions': len(questions),
        'difficulty_distribution': {
            'easy': len([q for q in questions if q['difficulty'] == 'easy']),
            'hard': len([q for q in questions if q['difficulty'] == 'hard'])
        },
        'questions_per_pattern': len(questions) / len(all_patterns[qa_type]) if all_patterns[qa_type] else 0,
        'sample_questions': [
            {
                'question': q['question'],
                'difficulty': q['difficulty'],
                'id': q['question_id']
            } for q in questions[:10]
        ]
    }
    
    analysis_file = qa_dir / 'question_analysis.json'
    with open(analysis_file, 'w', encoding='utf-8') as f:
        json.dump(analysis, f, ensure_ascii=False, indent=2)

# 전체 요약 저장
summary = {
    'total_questions': total_questions,
    'qa_type_distribution': {
        qa_type: len(questions) for qa_type, questions in all_questions.items()
    },
    'difficulty_distribution': {
        'easy': sum(len([q for q in questions if q['difficulty'] == 'easy']) for questions in all_questions.values()),
        'hard': sum(len([q for q in questions if q['difficulty'] == 'hard']) for questions in all_questions.values())
    },
    'generation_timestamp': pd.Timestamp.now().isoformat()
}

summary_file = output_path / 'question_generation_summary.json'
with open(summary_file, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"\n✓ 전체 요약: {summary_file}")
print(f"\n질문 생성 완료!")
print(f"총 {total_questions}개 질문이 생성되었습니다.")

## Cell 35: 생성된 질문 품질 검증

In [ ]:
# Phase 4의 생성된 질문 품질 검증
print("=== 생성된 질문 품질 검증 ===\n")

# 1. 기본 통계
print("1. 기본 통계:")
print(f"   총 질문 수: {total_questions}")
print(f"   평균 질문/패턴: {total_questions / total_patterns:.2f}")

# 2. 난이도 분포
print("\n2. 난이도 분포:")
easy_count = sum(len([q for q in questions if q['difficulty'] == 'easy']) for questions in all_questions.values())
hard_count = sum(len([q for q in questions if q['difficulty'] == 'hard']) for questions in all_questions.values())
print(f"   Easy: {easy_count} ({easy_count/total_questions*100:.1f}%)")
print(f"   Hard: {hard_count} ({hard_count/total_questions*100:.1f}%)")

# 3. QA 유형별 분석
print("\n3. QA 유형별 분석:")
for qa_type, questions in all_questions.items():
    print(f"\n   {qa_type}:")
    print(f"   - 질문 수: {len(questions)}")
    print(f"   - 패턴 수: {len(all_patterns[qa_type])}")
    print(f"   - 질문/패턴: {len(questions)/len(all_patterns[qa_type]):.2f}")
    
    # 난이도 분포
    easy = len([q for q in questions if q['difficulty'] == 'easy'])
    hard = len([q for q in questions if q['difficulty'] == 'hard'])
    print(f"   - Easy: {easy} ({easy/len(questions)*100:.1f}%)")
    print(f"   - Hard: {hard} ({hard/len(questions)*100:.1f}%)")

# 4. 질문 길이 분석
print("\n4. 질문 길이 분석:")
all_lengths = []
for qa_type, questions in all_questions.items():
    lengths = [len(q['question']) for q in questions]
    all_lengths.extend(lengths)
    print(f"   {qa_type}: 평균 {np.mean(lengths):.1f}자")

print(f"\n   전체 평균 길이: {np.mean(all_lengths):.1f}자")
print(f"   최소/최대: {min(all_lengths)}/{max(all_lengths)}자")

# 5. Hard 질문의 구조적 답변 유도 검증
print("\n5. Hard 질문의 구조적 답변 유도 검증:")
structural_indicators = ['→', '형식', '형태', '나열', '제시', '구분', '명시']

for qa_type in all_questions.keys():
    hard_questions = [q for q in all_questions[qa_type] if q['difficulty'] == 'hard']
    structural_count = 0
    
    for q in hard_questions:
        if any(indicator in q['question'] for indicator in structural_indicators):
            structural_count += 1
    
    if hard_questions:
        percentage = structural_count / len(hard_questions) * 100
        print(f"   {qa_type}: {structural_count}/{len(hard_questions)} ({percentage:.1f}%)")

# 6. 중복 질문 체크
print("\n6. 중복 질문 체크:")
all_question_texts = []
for questions in all_questions.values():
    all_question_texts.extend([q['question'] for q in questions])

unique_questions = set(all_question_texts)
duplicate_count = len(all_question_texts) - len(unique_questions)

print(f"   총 질문: {len(all_question_texts)}")
print(f"   고유 질문: {len(unique_questions)}")
print(f"   중복 질문: {duplicate_count}")

# 7. 템플릿 채움 오류 체크
print("\n7. 템플릿 채움 검증:")
template_errors = 0
for qa_type, questions in all_questions.items():
    for q in questions:
        if '{' in q['question'] or '}' in q['question']:
            template_errors += 1
            print(f"   오류: {q['question_id']} - {q['question']}")

if template_errors == 0:
    print("   ✓ 모든 템플릿이 정상적으로 채워짐")
else:
    print(f"   ✗ {template_errors}개 템플릿 채움 오류 발견")

# 8. 샘플 Hard 질문 출력 (구조적 답변 유도 확인)
print("\n8. 구조적 답변을 유도하는 Hard 질문 샘플:")
for qa_type in ['multi_hop', 'causal_chains']:
    print(f"\n   [{qa_type}]")
    hard_structural = [
        q for q in all_questions[qa_type] 
        if q['difficulty'] == 'hard' and any(ind in q['question'] for ind in ['→', '형식', '형태'])
    ]
    for q in hard_structural[:2]:
        print(f"   - {q['question']}")

print("\n✓ Phase 4 질문 품질 검증 완료!")

## Phase 4 최종 요약

Phase 4에서 성공적으로 템플릿 기반 질문 생성을 완료했습니다:

### 생성 결과
| QA 유형 | 생성된 질문 수 | Easy | Hard | Hard 비율 |
|---------|---------------|------|------|-----------|
| Multi-hop Reasoning | 214개 | 44개 | 170개 | 79.4% |
| Community Synthesis | 76개 | 12개 | 64개 | 84.2% |
| Cross-context Integration | 64개 | 28개 | 36개 | 56.2% |
| Implicit Relationships | 125개 | 29개 | 96개 | 76.8% |
| Causal Chains | 130개 | 32개 | 98개 | 75.4% |
| **총계** | **609개** | **145개** | **464개** | **76.2%** |

### 주요 특징
- **난이도 이원화**: Easy/Hard 구조
- **구조적 답변 유도**: Hard 질문 100%가 명확한 답변 형식 제시 (→, 형식, 형태 등)
- **품질 검증 완료**: 템플릿 채움 오류 0개, 중복 질문 0개

생성된 질문들은 각 QA 유형별 디렉토리의 `generated_questions.json`에 저장되었습니다.